# Vega 0.8B and 4B: an extensive typed-decision benchmark with ablations (Kaggle 2x T4)

What this notebook measures, on the same rows, with the same questions:

| readout | what answers | labelled examples needed |
|---|---|---|
| `base-*/generate` | the frozen backbone, asked in words, greedy decode | none |
| `base-*/logprob` | the frozen backbone, scoring each option as text | none |
| `vega-*/engine_no_adapter` | the physics engine with its adapters off | none |
| `vega-*/engine` | the engine with its gated adapters, which is what the published figures measure | none |
| `vega-*/engine_views2` | the same engine averaged over two surface forms | none |
| `vega-*/ttt_6 ... ttt_50` | a linear head on the same pooled features, a learning curve | 6 to 50 |
| `vega-*/auto` | the head where it beat chance in cross validation, the engine elsewhere | 6 to 50 |
| `jev/*` | the hosted reference, over the network, same request shape | none |

Three suites:

1. **43 public tasks** built from their own upstream datasets: intent (77 and 151 classes), topic,
   affect, entailment, verification, safety and prompt injection, legal, medical, science,
   commonsense, knowledge, code and two ordinal scales.
2. **JevBench public tiers** (MIT), which are already typed decisions and carry paraphrase twins,
   so option-order and wording robustness are measurable.
3. **DecisionBench** (43 tasks, three primitives, gold probability distributions as well as gold
   labels).

**Before you run.** Settings: Accelerator **GPU T4 x2**, Internet **on**, and two secrets
attached under *Add-ons -> Secrets*:

| secret | what it is for | without it |
|---|---|---|
| `JEV_API` | the key for the hosted reference at `api.typesafe.ai`. The notebook reads it with `UserSecretsClient().get_secret("JEV_API")` and never prints it. | the `jev/*` column is skipped with a printed reason and everything local still runs |
| `HF_TOKEN` | a Hugging Face read token, so the checkpoints and the gated datasets download without rate limiting | the public checkpoints still load anonymously; a few datasets may rate limit |

Then **Save Version -> Save & Run All (Commit)** so it runs in the background: the full menu is
about four and a half hours on 2x T4, mostly the 4B. Set `CFG["quick"] = True` first for a five
minute dry run that exercises every code path.

**What it writes.** One JSON at `/kaggle/working/vega_benchmark.json`, saved after every task, so a
run that hits the twelve hour wall, or dies, still leaves everything it finished.

**If a run does not finish.** Download that JSON from the failed version's output, attach it to the
next run as an input dataset, and set `CFG["resume_from"]` to its path. Every readout already in it
is carried forward and skipped, so only the unfinished phases cost anything. The memory line
printed at each phase boundary is the thing to read first if a kernel died: a kernel killed for
host memory leaves no traceback, only a dead kernel.

**What it deliberately does not do.**

* It does not fit anything on JevBench or DecisionBench rows. Those are other people's evaluation
  sets; the one thing that would spoil them is training on them. TTT heads are fitted only on the
  public tasks' own training splits, and never on a row that is then scored.
* It does not write dataset text into the output. The JSON holds row ids, metrics and
  probabilities. Several of these sets are non-commercial or research-only and are not ours to
  redistribute.
* It does not report an accuracy without its chance level. A 151-class task and a yes/no task do
  not belong in the same mean, so `skill` (chance-corrected) is the column to read across tasks.


## 0. What the last full run measured

This is the published table. It is what this notebook produced on 2026-10-11 on two Tesla T4s,
and it is here so that a run of your own has something to land next to. Every number below is
read straight out of `vega_benchmark.json`; nothing is typed in by hand.

| metric | Vega 0.8B | Vega 4B | Jev |
| --- | ---: | ---: | ---: |
| **Public menu, 43 tasks, macro average** | | | |
| accuracy | 0.556 | 0.717 | **0.836** |
| skill, chance corrected | 0.388 | 0.594 | **0.755** |
| macro F1 | 0.486 | 0.649 | **0.796** |
| expected calibration error | 0.151 | 0.148 | **0.115** |
| negative log likelihood | 1.099 | **0.768** | 0.878 |
| area under the risk coverage curve | 0.362 | 0.169 | **0.077** |
| latency per decision, p50 ms | **74** | 106 | 162 |
| **JevBench, 231 items** | | | |
| accuracy | 0.545 | 0.662 | **0.866** |
| accuracy, hard tier, 111 items | 0.414 | 0.513 | **0.730** |
| paraphrase consistency, 36 groups | 0.694 | 0.722 | **0.972** |
| expected calibration error | 0.178 | 0.169 | **0.067** |
| **DecisionBench, 1,075 items** | | | |
| accuracy | 0.509 | 0.633 | **0.752** |
| expected calibration error | 0.103 | **0.098** | 0.122 |
| negative log likelihood | 1.628 | **1.420** | 3.196 |
| KL to the gold distribution, 10 rows | 1.012 | **0.854** | 6.099 |
| coverage, rows answered | **1.000** | **1.000** | 0.997 |
| latency per decision, p50 ms | **85** | 245 | 163 |

Both backbones were measured on the same rows with the same prompts, no engine in the path:
`base-4b/generate` reaches 0.700 accuracy on the menu and 0.634 on DecisionBench, within noise of
the 4B engine, so read the engine's contribution as coverage, probabilities and latency rather
than as raw accuracy. `base-0.8b/generate` reaches 0.523 and 0.464.

**Cross-checking a run of your own.** Nothing above needs to be taken on faith:

1. Run this notebook with the defaults. `CFG["resume_from"]` and `CFG["redo"]` are empty, so it
   measures everything from scratch.
2. Compare `summary.public_macro` in your `vega_benchmark.json` with the public block above, and
   `results.jevbench.all` and `results.decisionbench.all` with the other two. The keys are the
   same strings used here.
3. Expect small movement and know where it comes from. The row sample is seeded by `CFG["seed"]`,
   so the same seed draws the same rows, but Jev is a live hosted endpoint whose weights can
   change under you, the `datasets` versions can reshuffle an upstream split, and the TTT heads
   are refitted per run. Local Vega readouts at a fixed seed and fixed library versions should
   reproduce to the row.
4. If a column is missing rather than different, read `notes` and `phases`: a skipped Jev column
   means no `JEV_API` secret, and a missing base column means the backbone was not loaded.
5. `environment` records python, torch, transformers, vegaml, numpy and both GPUs. A number that
   differs by more than a couple of rows with the same seed is worth reporting with that block
   attached.


## 1. Configuration

In [ ]:
# Every knob the run has. The defaults are sized for a 12 hour Kaggle commit on 2x T4.
CFG = {
    "seed": 20261011,
    "models": ["0.8b", "4b"],            # both sizes. Drop one to roughly halve the run
    "n_eval": 60,                        # evaluation rows per public task
    "n_fit": 50,                         # labelled rows a head may use, never from the eval rows
    "shots": [6, 12, 25, 50],            # head budgets, fitted on prefixes of the same features
    "folds": 5,
    "task_keys": None,                   # None = every task in the registry
    "views": [2],                        # engine surface-form averaging, on a subset of tasks
    "views_tasks": ["banking77", "anli_r3", "boolq", "casehold", "emotion", "tweet_hate",
                    "arc_challenge", "medmcqa"],
    "flip_tasks": ["banking77", "emotion", "anli_r3", "arc_challenge", "casehold", "mmlu",
                   "trec_coarse", "vitaminc", "climate_fever", "ledgar"],
    "decisionbench_per_task": 25,        # 43 tasks, so 25 is about 1,075 rows
    "jevbench": True,
    "jev_model": "jev-latest",           # the published board uses jev-1.13.0
    "jev_workers": 6,
    "jev_max_rps": None,                 # set a number if the endpoint rate limits
    "jev_on_public": True,
    "jev_on_decisionbench": True,
    "run_no_adapter": True,
    "run_ttt": True,
    "run_base_logprob": True,
    "run_base_generate": True,
    "base_max_len": 3072,                # prompt budget for the base-model baselines
    "base_batch": 16,                    # options scored per forward pass
    "gen_tokens": 12,
    # Qwen3.5's chat template opens a reasoning block, and the two sizes default the other way
    # round: unset, the 0.8B template closes it and the 4B leaves it open, which spends the whole
    # generation budget thinking and scored the 4B baseline at a parse rate of 0.00 on 26 of 43
    # tasks in the first full run. Off means both sizes answer immediately.
    "base_thinking": False,
    "budget_hours": 10.5,                # Kaggle stops a committed notebook at 12
    "base_min_free_gb": 6.0,             # refuse a second backbone copy below this much host RAM
    "free_vega_before_base": True,       # the base phase is last, so the engines can go first
    "out_path": "/kaggle/working/vega_benchmark.json",
    # Point this at a previous run's JSON, attached as an input dataset, and every readout it
    # already holds is skipped. That is the way back from a kernel that died mid-run: the file is
    # written after every task, so only the unfinished phases are paid for again.
    "resume_from": "",                   # e.g. "/kaggle/input/vega-bench-partial/vega_benchmark.json"
    # Readouts to forget from the resumed file and measure again, for when a carried-forward
    # number turns out to have been measured wrongly. Without this the resume guard protects it.
    "redo": [],                          # e.g. ["base-4b/generate"]
    # A previous run's jev_cache.jsonl, attached as an input. The cache is keyed by request hash,
    # and a fresh run with the same seed sends the same requests, so seeding it makes the hosted
    # column free instead of paying for 2,767 decisions again.
    "jev_cache_from": "",                # e.g. "/kaggle/input/vega-bench-partial/jev_cache.jsonl"
    "quick": False,                      # True: a few rows, 0.8B only, every code path, ~5 minutes
}

if CFG["quick"]:
    CFG.update(models=["0.8b"], n_eval=8, n_fit=12, shots=[6, 12], decisionbench_per_task=2,
               task_keys=["boolq", "emotion", "arc_challenge", "banking77"], views=[],
               views_tasks=[], flip_tasks=["emotion"], budget_hours=0.6,
               out_path="/kaggle/working/vega_benchmark_quick.json")
print(CFG)

## 2. Environment

`transformers` 5.17 or newer is required: the backbone is a Qwen3.5 and older versions do not know
the architecture. `vegaml` brings the inference code; the checkpoint repository holds weights and
config only and no Python is ever downloaded from it.

In [ ]:
!pip -q install -U "vegaml>=0.8.0" "transformers>=5.17" "datasets>=4.0" "huggingface_hub>=1.5" 2>&1 | tail -3

In [ ]:
import os

# The HF cache goes to /kaggle/temp, not /kaggle/working: the 4B backbone alone is about 8 GB and
# /kaggle/working is the 20 GB output volume that gets committed with the notebook.
os.environ.setdefault("HF_HOME", "/kaggle/temp/hf")
os.environ.setdefault("HF_DATASETS_CACHE", "/kaggle/temp/hf/datasets")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.makedirs("/kaggle/temp/hf", exist_ok=True)

JEV_KEY, SECRETS = None, None
try:
    from kaggle_secrets import UserSecretsClient
    SECRETS = UserSecretsClient()
    JEV_KEY = (SECRETS.get_secret("JEV_API") or "").strip()
    print("JEV_API secret: found, %d characters" % len(JEV_KEY))
except Exception as exc:
    print("no JEV_API secret (%s: %s); the Jev column will be skipped"
          % (type(exc).__name__, str(exc)[:120]))

if SECRETS is not None:            # optional, and only needed for a private checkpoint
    try:
        tok = (SECRETS.get_secret("HF_TOKEN") or "").strip()
        if tok:
            os.environ["HF_TOKEN"] = tok
            print("HF_TOKEN secret: found")
    except Exception:
        print("no HF_TOKEN secret; the public checkpoints load anonymously")

# The key stays in memory and in the request header. It is never written to the output JSON, never
# printed, and the Jev response cache on disk holds request hashes rather than requests.
print("key in environment:", "JEV_API" in os.environ)

## 3. The harness

Seven modules, written out so the notebook is self-contained.

In [ ]:
%%writefile bench_common.py
"""Metrics, typed-question helpers and the prompt-budget audit for the Vega benchmark run.

Everything here is plain Python and numpy so it can be unit tested off the GPU. Three rules hold
throughout:

* A readout that cannot answer scores as wrong, and the share of answered rows is reported as
  `coverage`. Dropping unanswerable rows would flatter whichever readout fails loudest.
* Probabilities are kept, not just the argmax, so calibration is measurable. A readout that only
  emits a label gets `ece = None` rather than a fabricated zero.
* No source text leaves this harness. The saved JSON holds row ids, metrics and probabilities. The
  evaluation sets carry other people's licences and are not ours to redistribute.
"""
import json
import math
import os
import sys
import time

import numpy as np

EPS = 1e-12


# ----------------------------------------------------------------------------- typed questions
def label_ids(q):
    """The option ids a readout must span, for any of the three answer types."""
    t = q["type"]
    if t == "boolean":
        return ["false", "true"]
    if t == "score":
        return [str(i) for i in range(len(q.get("criteria") or []))]
    return [str(k) for k in q["criteria"]]


def option_texts(q):
    """What each option looks like in a prompt: the id plus its criterion when one is given."""
    t, crit = q["type"], q.get("criteria")
    if t == "choice":
        return [str(k) if not v else "%s - %s" % (k, v) for k, v in crit.items()]
    if t == "score":
        return ["level %d - %s" % (i, c) for i, c in enumerate(crit or [])]
    crit = crit or {}
    return ["false - " + (crit.get("false") or "no, the statement does not hold"),
            "true - " + (crit.get("true") or "yes, the statement holds")]


def serialize_state(state):
    return state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)


def chance_level(q):
    """The accuracy a uniform guess reaches, which is what a many-class score has to be read against."""
    return 1.0 / max(len(label_ids(q)), 2)


# ----------------------------------------------------------------------------- budget audit
def budget_audit(tok, questions, opt_max_tokens=32, ins_max_tokens=256):
    """How much of each question survives Vega's option and instruction caps.

    `build_prompt` truncates an option at `opt_max_tokens` and an instruction at `ins_max_tokens`
    rather than refusing, so a task with long options is quietly asked a different question than the
    one on paper. Worse, two options that share a long prefix can truncate to the same tokens, and
    then the task is unanswerable by construction. Both are measured here and reported per task, so
    a low score on CaseHOLD or MMLU-Pro is read as a budget limit rather than as reasoning.
    """
    n_opt = n_over = n_collide = 0
    ins_over = 0
    opt_lens, ins_lens = [], []
    for q in questions:
        ins = tok(str(q["instructions"]), add_special_tokens=False)["input_ids"]
        ins_lens.append(len(ins))
        ins_over += int(len(ins) > ins_max_tokens)
        seen = {}
        for text in option_texts(q):
            ids = tok(text, add_special_tokens=False)["input_ids"]
            opt_lens.append(len(ids))
            n_opt += 1
            n_over += int(len(ids) > opt_max_tokens)
            key = tuple(ids[:opt_max_tokens])
            seen[key] = seen.get(key, 0) + 1
        n_collide += sum(v - 1 for v in seen.values() if v > 1)
    pct = lambda a, b: round(a / b, 4) if b else 0.0
    return {"questions": len(questions), "options": n_opt,
            "options_over_cap": n_over, "options_over_cap_frac": pct(n_over, n_opt),
            "option_tokens_p50": int(np.percentile(opt_lens, 50)) if opt_lens else 0,
            "option_tokens_p95": int(np.percentile(opt_lens, 95)) if opt_lens else 0,
            "option_tokens_max": int(max(opt_lens)) if opt_lens else 0,
            "truncation_collisions": n_collide,
            "truncation_collisions_frac": pct(n_collide, n_opt),
            "instructions_over_cap": ins_over,
            "instruction_tokens_p95": int(np.percentile(ins_lens, 95)) if ins_lens else 0}


# ----------------------------------------------------------------------------- metrics
def _vec(probs, labels):
    """A probability vector over `labels`, renormalised, with missing options read as zero."""
    p = np.array([float(probs.get(k, 0.0)) for k in labels], dtype=np.float64)
    p = np.clip(p, 0.0, None)
    s = p.sum()
    return p / s if s > 0 else np.full(len(labels), 1.0 / max(len(labels), 1))


def ece(conf, correct, bins=15):
    """Expected calibration error on the top-label confidence, equal-width bins."""
    conf, correct = np.asarray(conf, dtype=np.float64), np.asarray(correct, dtype=np.float64)
    if conf.size == 0:
        return None
    edges = np.linspace(0.0, 1.0, bins + 1)
    out = 0.0
    for i in range(bins):
        lo, hi = edges[i], edges[i + 1]
        sel = (conf > lo) & (conf <= hi) if i else (conf >= lo) & (conf <= hi)
        if not sel.any():
            continue
        out += sel.mean() * abs(correct[sel].mean() - conf[sel].mean())
    return float(out)


def aurc(conf, correct):
    """Area under the risk-coverage curve: the error rate you keep as you drop the least confident
    answers first. Lower is better, and unlike accuracy it rewards knowing which answers to doubt."""
    conf, correct = np.asarray(conf, dtype=np.float64), np.asarray(correct, dtype=np.float64)
    if conf.size == 0:
        return None
    order = np.argsort(-conf)
    err = 1.0 - correct[order]
    risks = np.cumsum(err) / np.arange(1, err.size + 1)
    return float(risks.mean())


def macro_f1(gold, pred, labels):
    f1s = []
    for lab in labels:
        tp = sum(1 for g, p in zip(gold, pred) if g == lab and p == lab)
        fp = sum(1 for g, p in zip(gold, pred) if g != lab and p == lab)
        fn = sum(1 for g, p in zip(gold, pred) if g == lab and p != lab)
        if tp + fp + fn == 0:
            continue
        prec = tp / (tp + fp) if tp + fp else 0.0
        rec = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * prec * rec / (prec + rec) if prec + rec else 0.0)
    return float(np.mean(f1s)) if f1s else None


def score_metrics(records, labels=None, qtype="choice", chance=None):
    """Every metric the suite reports, from a list of per-row records.

    A record is {"gold": id, "pred": id or None, "probs": {id: p} or None, "latency_ms": float,
    "abstain": bool or None, "set_size": int or None}. `pred = None` means the readout could not
    answer, which counts as wrong and lowers coverage.

    `labels` is the option set when every row shares one. Rows whose option set is their own (one
    DecisionBench row offers "Negligible/Low/Moderate", the next offers something else) carry
    "labels" and "chance" on the record, and then the distribution metrics are computed row by row
    and the label-set metrics (macro F1) are left out rather than computed over a union that no
    single row was asked.
    """
    n = len(records)
    if not n:
        return {"n": 0}
    answered = [r for r in records if r.get("pred") is not None]
    correct_all = [1.0 if r.get("pred") == r["gold"] else 0.0 for r in records]
    acc = float(np.mean(correct_all))
    row_labels = [r.get("labels") or labels or [] for r in records]
    chances = [r.get("chance", 1.0 / max(len(ls), 2)) for r, ls in zip(records, row_labels)]
    ch = float(np.mean(chances)) if chance is None else float(chance)
    out = {"n": n, "answered": len(answered), "coverage": round(len(answered) / n, 4),
           "accuracy": round(acc, 4), "chance": round(ch, 4),
           "skill": round(max(0.0, (acc - ch) / (1.0 - ch)) if ch < 1 else 0.0, 4)}
    if labels:
        out["macro_f1"] = macro_f1([r["gold"] for r in records],
                                   [r.get("pred") for r in records], labels)
    lat = [r["latency_ms"] for r in records if r.get("latency_ms") is not None]
    if lat:
        out.update(latency_ms_p50=round(float(np.percentile(lat, 50)), 1),
                   latency_ms_p95=round(float(np.percentile(lat, 95)), 1),
                   latency_ms_mean=round(float(np.mean(lat)), 1))
    withp = [(r, ls) for r, ls in zip(records, row_labels) if r.get("probs") and ls]
    if withp:
        conf, corr, nlls, briers = [], [], [], []
        for r, ls in withp:
            p = _vec(r["probs"], ls)
            top = int(p.argmax())
            conf.append(float(p[top]))
            corr.append(1.0 if ls[top] == r["gold"] else 0.0)
            if r["gold"] in ls:
                gi = ls.index(r["gold"])
                nlls.append(-math.log(max(float(p[gi]), EPS)))
                onehot = np.zeros(len(ls))
                onehot[gi] = 1.0
                briers.append(float(((p - onehot) ** 2).sum()))
        out.update(ece=round(ece(conf, corr), 4),
                   nll=round(float(np.mean(nlls)), 4) if nlls else None,
                   brier=round(float(np.mean(briers)), 4) if briers else None,
                   aurc=round(aurc(conf, corr), 4),
                   mean_confidence=round(float(np.mean(conf)), 4),
                   rows_with_probabilities=len(withp))
    if qtype == "score" and labels:
        idx = {lab: i for i, lab in enumerate(labels)}
        pairs = [(idx.get(r["gold"]), idx.get(r.get("pred"))) for r in records]
        pairs = [(g, p) for g, p in pairs if g is not None and p is not None]
        if pairs:
            out["ordinal_mae"] = round(float(np.mean([abs(g - p) for g, p in pairs])), 4)
    ab = [r["abstain"] for r in records if r.get("abstain") is not None]
    if ab:
        out["abstain_rate"] = round(float(np.mean(ab)), 4)
        kept = [r for r in records if not r.get("abstain")]
        if kept:
            out["accuracy_when_not_abstaining"] = round(
                float(np.mean([1.0 if r.get("pred") == r["gold"] else 0.0 for r in kept])), 4)
    sets = [r["set_size"] for r in records if r.get("set_size")]
    if sets:
        out["conformal_set_size_mean"] = round(float(np.mean(sets)), 3)
        hit = [1.0 if r.get("set_hit") else 0.0 for r in records if r.get("set_size")]
        out["conformal_coverage"] = round(float(np.mean(hit)), 4)
    err = [r for r in records if r.get("error")]
    if err:
        out["rows_with_errors"] = len(err)
        out["first_error"] = str(err[0]["error"])[:200]
    trunc = [r for r in records if r.get("state_truncated")]
    if trunc:
        out["rows_state_truncated"] = len(trunc)
    return out


def trivial_baselines(golds, labels):
    """What majority class and a uniform guess score on the same rows. Any readout below these on a
    task has told you nothing, and on a 77-class task the difference is easy to miss by eye."""
    if not golds:
        return {}
    counts = {lab: golds.count(lab) for lab in set(golds)}
    top = max(counts, key=counts.get)
    k = max(len(labels), 2)
    return {"majority_class": top, "majority_accuracy": round(counts[top] / len(golds), 4),
            "uniform_accuracy": round(1.0 / k, 4),
            "uniform_nll": round(float(math.log(k)), 4),
            "label_entropy": round(float(-sum((c / len(golds)) * math.log(c / len(golds))
                                              for c in counts.values())), 4)}


# ----------------------------------------------------------------------------- misc
class Budget:
    """A wall-clock guard. Kaggle stops a committed notebook at 12 hours, and a run that dies at
    hour 11 with nothing written is a wasted day, so every phase checks this and stops early."""

    def __init__(self, seconds):
        self.t0, self.limit = time.time(), float(seconds)

    def left(self):
        return self.limit - (time.time() - self.t0)

    def spent(self):
        return time.time() - self.t0

    def ok(self, need=0.0):
        return self.left() > need


# ----------------------------------------------------------------------------- host memory
def memory_gb():
    """This process's resident size and how much room the container has left, in GB.

    A Kaggle kernel that outgrows the machine is killed outright: the notebook log shows
    "Kernel died while waiting for execute reply" and no Python traceback, because the process
    never got to raise anything. The only way to see that coming is to record these numbers
    before and after every model load, so this is called at every phase boundary.
    """
    out = {"rss_gb": None, "available_gb": None, "total_gb": None,
           "cgroup_used_gb": None, "cgroup_limit_gb": None, "cgroup_cache_gb": None,
           "cgroup_committed_gb": None}
    try:
        with open("/proc/self/status") as fh:
            for line in fh:
                if line.startswith("VmRSS:"):
                    out["rss_gb"] = round(int(line.split()[1]) / 1048576.0, 2)
                    break
    except OSError:
        pass
    try:
        with open("/proc/meminfo") as fh:
            for line in fh:
                if line.startswith("MemAvailable:"):
                    out["available_gb"] = round(int(line.split()[1]) / 1048576.0, 2)
                elif line.startswith("MemTotal:"):
                    out["total_gb"] = round(int(line.split()[1]) / 1048576.0, 2)
    except OSError:
        pass
    if out["rss_gb"] is None:                      # not Linux: good enough for a dry run
        try:
            import resource
            raw = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
            # ru_maxrss is bytes on darwin and kilobytes on linux, and it is a peak rather than
            # the current size, so this branch is only a sanity figure for the dry run
            out["rss_gb"] = round(raw / (1073741824.0 if sys.platform == "darwin"
                                         else 1048576.0), 2)
        except Exception:
            pass
    out.update(_cgroup_memory())
    return out


def _cgroup_memory():
    """The container's own accounting, which is what the kill decision uses.

    /proc/meminfo reports the host, and on a shared machine the host can look roomy while this
    container is one allocation from being killed. cgroup v2 first, then v1.

    `used` counts the page cache this container caused, and reading an 11 GB checkpoint charges
    all of it. That cache is clean and reclaimable, so `limit - used` is not the room left: on
    the resumed run it read 3.4 GB while the host still had 25 GB available, and a load that
    would have fitted was refused. `committed` takes the cache back out and is the number to
    compare an allocation against.
    """
    v2 = ("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory.max",
          "/sys/fs/cgroup/memory.stat", ("file", "slab_reclaimable"))
    v1 = ("/sys/fs/cgroup/memory/memory.usage_in_bytes",
          "/sys/fs/cgroup/memory/memory.limit_in_bytes",
          "/sys/fs/cgroup/memory/memory.stat", ("total_cache",))
    for cur, mx, stat, cache_keys in (v2, v1):
        try:
            used = int(open(cur).read().strip()) / 1073741824.0
        except (OSError, ValueError):
            continue
        limit = None
        try:
            raw = open(mx).read().strip()
            if raw != "max":
                val = int(raw)
                if val < 2 ** 50:                  # v1 writes a sentinel near 2**63 for "no limit"
                    limit = round(val / 1073741824.0, 2)
        except (OSError, ValueError):
            pass
        cache = None
        try:
            fields = {}
            for line in open(stat):
                parts = line.split()
                if len(parts) == 2:
                    fields[parts[0]] = int(parts[1])
            if any(k in fields for k in cache_keys):
                cache = sum(fields.get(k, 0) for k in cache_keys) / 1073741824.0
        except (OSError, ValueError):
            pass
        return {"cgroup_used_gb": round(used, 2), "cgroup_limit_gb": limit,
                "cgroup_cache_gb": None if cache is None else round(cache, 2),
                "cgroup_committed_gb": None if cache is None
                else round(max(used - cache, 0.0), 2)}
    return {}


def headroom_gb():
    """GB that can still be allocated before the kernel is killed, or None when unknowable.

    The smaller of what the container has left and what the host has available: either one
    running out ends the process. The container's own figure counts out its reclaimable page
    cache, since the kernel drops that rather than killing the process, and only falls back to
    the raw usage when the cgroup does not break it down.
    """
    m = memory_gb()
    room = []
    if m["cgroup_limit_gb"] is not None:
        used = m["cgroup_committed_gb"]
        if used is None:
            used = m["cgroup_used_gb"]
        if used is not None:
            room.append(m["cgroup_limit_gb"] - used)
    if m["available_gb"] is not None:
        room.append(m["available_gb"])
    return round(min(room), 2) if room else None


def memory_line(label=""):
    m = memory_gb()
    head = headroom_gb()
    return ("%srss %s GB, container %s of %s GB (%s cache, %s committed), host available %s GB, "
            "headroom %s GB"
            % (label + ": " if label else "", m["rss_gb"], m["cgroup_used_gb"],
               m["cgroup_limit_gb"], m["cgroup_cache_gb"], m["cgroup_committed_gb"],
               m["available_gb"], head))


def drop_file_cache(*paths):
    """Ask the kernel to forget the page cache for files under these paths.

    Reading the two backbones charges about 11 GB of clean page cache to the container, and the
    weights are on the GPU afterwards, so nothing needs those pages again. Dropping them lowers
    the number the kill decision reads. Best effort: pages still mapped stay, and this is a hint
    to the kernel rather than a command, so the caller reports the before and after rather than
    assuming an amount.
    """
    freed = 0
    if not hasattr(os, "posix_fadvise"):
        return 0
    for root in paths:
        if not root or not os.path.exists(root):
            continue
        walk = [(root, [], [os.path.basename(root)])] if os.path.isfile(root) else os.walk(root)
        base = os.path.dirname(root) if os.path.isfile(root) else None
        for folder, _, files in walk:
            for name in files:
                full = os.path.join(base if base is not None else folder, name)
                if not full.endswith((".safetensors", ".bin", ".pt", ".gguf")):
                    continue
                try:
                    fd = os.open(full, os.O_RDONLY)
                except OSError:
                    continue
                try:
                    size = os.fstat(fd).st_size
                    os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
                    freed += size
                except OSError:
                    pass
                finally:
                    os.close(fd)
    return round(freed / 1073741824.0, 2)


def jsonable(obj):
    """numpy scalars and arrays are not JSON serialisable, and silently stringifying them later
    makes the output unusable for plots. Convert once, here."""
    if isinstance(obj, dict):
        return {str(k): jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [jsonable(v) for v in obj]
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return jsonable(obj.tolist())
    if isinstance(obj, (bool, int, float, str)) or obj is None:
        return obj
    return str(obj)


In [ ]:
%%writefile bench_tasks.py
"""The public evaluation tasks, each expressed as a typed decision Vega and Jev can both be asked.

Three shapes cover everything here:

* **fixed-option** tasks (intent, topic, sentiment, safety): one question per task, the options are
  the label set, the row supplies the state. 77-way and 151-way tasks are in on purpose, because a
  wide label space with few examples per class is where a fitted head stops helping.
* **per-row-option** tasks (multiple choice): the options differ row by row, so the question is
  built per row and the option **ids** are kept stable (A, B, C ...) so one head can span them.
* **ordinal** tasks: answered as `score`, which is the only type that carries an order, so an
  ordinal mistake of one level is not scored the same as a mistake of three.

Each entry records its upstream licence. Several are non-commercial or research-only, which is why
this harness evaluates on them and never writes their text into its output.
"""
import hashlib
import json
import random

CHOICE_IDS = [chr(ord("A") + i) for i in range(26)]


# ----------------------------------------------------------------------------- helpers
def _names(ds, col):
    f = ds.features.get(col)
    return list(getattr(f, "names", []) or [])


def _text(row, cols):
    return {c: row[c] for c in cols if row.get(c) is not None}


def fixed(key, hf, config, label_col, state_cols, instructions, licence,
          qtype="choice", criteria=None, labels=None, eval_split="test", train_split="train",
          gold_map=None, state_fn=None, group=None, note=None):
    """A task whose option set is the same for every row."""
    return {"key": key, "hf": hf, "config": config, "kind": "fixed", "qtype": qtype,
            "label_col": label_col, "state_cols": state_cols, "instructions": instructions,
            "licence": licence, "criteria": criteria, "labels": labels,
            "eval_split": eval_split, "train_split": train_split, "gold_map": gold_map,
            "state_fn": state_fn, "group": group or "classification", "note": note}


def perrow(key, hf, config, instructions, licence, state_fn, options_fn, gold_fn,
           eval_split="test", train_split="train", group="multiple_choice", note=None,
           qtype="choice"):
    """A multiple-choice task: options come from the row, ids stay A, B, C ... so a head can span them."""
    return {"key": key, "hf": hf, "config": config, "kind": "perrow", "qtype": qtype,
            "instructions": instructions, "licence": licence, "state_fn": state_fn,
            "options_fn": options_fn, "gold_fn": gold_fn, "eval_split": eval_split,
            "train_split": train_split, "group": group, "note": note}


# ----------------------------------------------------------------------------- the registry
YESNO = {"false": "no", "true": "yes"}

TASKS = [
    # ------------------------------------------------- intent and topic, wide label spaces
    fixed("banking77", "mteb/banking77", None, "label_text", ["text"],
          "Which banking intent does this customer utterance express?", "cc-by-4.0",
          group="intent", note="77 intents: the wide-label-space case. The mteb mirror is used "
                               "because PolyAI/banking77 still ships a loading script"),
    fixed("clinc150_oos", "clinc/clinc_oos", "plus", "intent", ["text"],
          "Which intent does this utterance express, or is it out of scope?", "cc-by-3.0",
          group="intent", note="150 intents plus an out-of-scope class"),
    fixed("massive_intent_en", "mteb/amazon_massive_intent", "en", "label_text", ["text"],
          "Which assistant intent does this utterance express?", "apache-2.0",
          group="intent", note="60 intents, spoken-assistant phrasing"),
    fixed("ledgar", "coastalcph/lex_glue", "ledgar", "label", ["text"],
          "Which contract provision type is this clause?", "cc-by-4.0",
          group="legal", note="100 provision types"),
    fixed("ag_news", "fancyzhx/ag_news", None, "label", ["text"],
          "Which section of the newspaper does this story belong to?", "unknown-see-card",
          group="topic"),
    fixed("dbpedia14", "fancyzhx/dbpedia_14", "dbpedia_14", "label", ["title", "content"],
          "Which ontology class does this article describe?", "cc-by-sa-3.0", group="topic"),
    fixed("trec_coarse", "CogComp/trec", None, "coarse_label", ["text"],
          "What kind of answer does this question ask for?", "unknown-see-card", group="topic"),
    fixed("emotion", "dair-ai/emotion", "split", "label", ["text"],
          "Which emotion does the writer express?", "other-see-card", group="affect"),

    # ------------------------------------------------- entailment, verification, policy
    fixed("anli_r3", "facebook/anli", "plain_text", "label", ["premise", "hypothesis"],
          "Does the premise entail the hypothesis, contradict it, or neither?", "cc-by-nc-4.0",
          eval_split="test_r3", train_split="train_r3", group="entailment",
          note="adversarial NLI, round 3"),
    fixed("glue_rte", "nyu-mll/glue", "rte", "label", ["sentence1", "sentence2"],
          "Does the first sentence entail the second?", "other-see-card",
          qtype="boolean", criteria={"false": "it does not entail it",
                                     "true": "it entails it"},
          labels=["true", "false"], eval_split="validation", group="entailment",
          gold_map={"entailment": "true", "not_entailment": "false"}),
    fixed("superglue_cb", "aps/super_glue", "cb", "label", ["premise", "hypothesis"],
          "Does the premise entail the hypothesis, contradict it, or neither?", "other-see-card",
          eval_split="validation", group="entailment"),
    fixed("vitaminc", "tals/vitaminc", None, "label", ["claim", "evidence"],
          "Does the evidence support the claim, refute it, or say too little to tell?",
          "cc-by-sa-3.0", group="verification",
          labels=["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]),
    fixed("climate_fever", "tdiggelm/climate_fever", None, "claim_label", ["claim"],
          "Does the evidence support the claim, refute it, say too little, or disagree with itself?",
          "unknown-see-card", eval_split="test", train_split="test", group="verification",
          state_fn=lambda row, ctx: {
              "claim": row["claim"],
              "evidence": [e["evidence"] for e in (row.get("evidences") or [])][:5]},
          note="one split upstream, so eval and fit rows are disjoint halves"),
    fixed("boolq", "google/boolq", None, "answer", ["passage", "question"],
          "Is the answer to the question yes, according to the passage?", "cc-by-sa-3.0",
          qtype="boolean", criteria=YESNO, labels=["true", "false"], eval_split="validation",
          group="reading", gold_map={True: "true", False: "false", "True": "true", "False": "false"}),

    # ------------------------------------------------- safety, abuse, injection
    fixed("toxic_chat_toxicity", "lmsys/toxic-chat", "toxicchat0124", "toxicity", ["user_input"],
          "Is this user message toxic?", "cc-by-nc-4.0", qtype="boolean",
          criteria={"false": "ordinary, even if blunt or rude",
                    "true": "hateful, harassing, sexual, violent or otherwise toxic"},
          labels=["true", "false"], group="safety",
          gold_map={1: "true", 0: "false", "1": "true", "0": "false"}),
    fixed("toxic_chat_jailbreak", "lmsys/toxic-chat", "toxicchat0124", "jailbreaking",
          ["user_input"], "Is this user message trying to jailbreak the assistant?",
          "cc-by-nc-4.0", qtype="boolean",
          criteria={"false": "an ordinary request, however unusual",
                    "true": "it tries to override the assistant's rules or role"},
          labels=["true", "false"], group="safety",
          gold_map={1: "true", 0: "false", "1": "true", "0": "false"}),
    fixed("aegis_prompt_safety", "nvidia/Aegis-AI-Content-Safety-Dataset-2.0", None,
          "prompt_label", ["prompt"], "Is this prompt unsafe to answer as written?", "cc-by-4.0",
          qtype="boolean", criteria={"false": "safe to answer", "true": "unsafe to answer"},
          labels=["true", "false"], group="safety",
          gold_map={"unsafe": "true", "safe": "false"}),
    fixed("prompt_injections", "deepset/prompt-injections", None, "label", ["text"],
          "Is this input a prompt injection attempt?", "apache-2.0", qtype="boolean",
          criteria={"false": "an ordinary request or statement",
                    "true": "it tries to override the instructions the system was given"},
          labels=["true", "false"], group="safety",
          gold_map={1: "true", 0: "false", "1": "true", "0": "false"}),
    fixed("jailbreak_cls", "jackhhao/jailbreak-classification", None, "type", ["prompt"],
          "Is this prompt a jailbreak attempt?", "apache-2.0", qtype="boolean",
          criteria={"false": "a benign prompt", "true": "a jailbreak attempt"},
          labels=["true", "false"], group="safety",
          gold_map={"jailbreak": "true", "benign": "false"}),
    fixed("enron_spam", "SetFit/enron_spam", None, "label_text", ["subject", "message"],
          "Is this email spam?", "unknown-see-card", qtype="boolean",
          criteria={"false": "wanted mail", "true": "unsolicited bulk mail"},
          labels=["true", "false"], group="safety",
          gold_map={"spam": "true", "ham": "false"}),
    fixed("unfair_tos", "coastalcph/lex_glue", "unfair_tos", "labels", ["text"],
          "Does this terms-of-service clause contain a potentially unfair term?", "cc-by-4.0",
          qtype="boolean", criteria={"false": "no unfair term", "true": "at least one unfair term"},
          labels=["true", "false"], group="legal",
          state_fn=lambda row, ctx: {"clause": row["text"]},
          gold_map="nonempty_list"),
    fixed("tweet_hate", "cardiffnlp/tweet_eval", "hate", "label", ["text"],
          "Is this tweet hate speech?", "unknown-see-card", qtype="boolean",
          criteria={"false": "not hate speech", "true": "hate speech"},
          labels=["true", "false"], group="affect",
          gold_map={1: "true", 0: "false"}),
    fixed("tweet_offensive", "cardiffnlp/tweet_eval", "offensive", "label", ["text"],
          "Is this tweet offensive?", "unknown-see-card", qtype="boolean",
          criteria={"false": "not offensive", "true": "offensive"},
          labels=["true", "false"], group="affect", gold_map={1: "true", 0: "false"}),
    fixed("tweet_irony", "cardiffnlp/tweet_eval", "irony", "label", ["text"],
          "Is this tweet ironic?", "unknown-see-card", qtype="boolean",
          criteria={"false": "said straight", "true": "ironic"},
          labels=["true", "false"], group="affect", gold_map={1: "true", 0: "false"}),
    fixed("tweet_stance_abortion", "cardiffnlp/tweet_eval", "stance_abortion", "label", ["text"],
          "What stance does this tweet take on the topic?", "unknown-see-card", group="affect"),

    # ------------------------------------------------- ordinal, answered as a score
    fixed("tweet_sentiment", "cardiffnlp/tweet_eval", "sentiment", "label", ["text"],
          "How positive is this tweet?", "unknown-see-card", qtype="score",
          criteria=["negative", "neutral", "positive"], group="ordinal",
          gold_map={0: "0", 1: "1", 2: "2"}),
    fixed("yelp_stars", "Yelp/yelp_review_full", "yelp_review_full", "label", ["text"],
          "How many stars does this review give?", "yelp-tos-research-only", qtype="score",
          criteria=["one star, very negative", "two stars", "three stars", "four stars",
                    "five stars, very positive"], group="ordinal",
          note="five ordered levels: the widest ordinal scale here",
          gold_map={i: str(i) for i in range(5)}),

    # ------------------------------------------------- multiple choice, per-row options
    perrow("arc_challenge", "allenai/ai2_arc", "ARC-Challenge",
           "Which option answers the question correctly?", "cc-by-sa-4.0",
           lambda r, c: {"question": r["question"]},
           lambda r, c: list(r["choices"]["text"]),
           lambda r, c: _letter_gold(r["answerKey"], list(r["choices"]["label"])),
           group="science"),
    perrow("arc_easy", "allenai/ai2_arc", "ARC-Easy",
           "Which option answers the question correctly?", "cc-by-sa-4.0",
           lambda r, c: {"question": r["question"]},
           lambda r, c: list(r["choices"]["text"]),
           lambda r, c: _letter_gold(r["answerKey"], list(r["choices"]["label"])),
           group="science"),
    perrow("openbookqa", "allenai/openbookqa", "main",
           "Which option answers the question correctly?", "unknown-see-card",
           lambda r, c: {"question": r["question_stem"]},
           lambda r, c: list(r["choices"]["text"]),
           lambda r, c: _letter_gold(r["answerKey"], list(r["choices"]["label"])),
           group="science"),
    perrow("sciq", "allenai/sciq", None,
           "Which option answers the science question correctly?", "cc-by-nc-3.0",
           lambda r, c: {"question": r["question"]},
           lambda r, c: _shuffled(c, [r["correct_answer"], r["distractor1"], r["distractor2"],
                                      r["distractor3"]])[0],
           lambda r, c: _shuffled(c, [r["correct_answer"], r["distractor1"], r["distractor2"],
                                      r["distractor3"]])[1],
           group="science"),
    perrow("commonsenseqa", "tau/commonsense_qa", None,
           "Which option answers the question correctly?", "mit",
           lambda r, c: {"question": r["question"]},
           lambda r, c: list(r["choices"]["text"]),
           lambda r, c: _letter_gold(r["answerKey"], list(r["choices"]["label"])),
           eval_split="validation", group="commonsense"),
    perrow("hellaswag", "Rowan/hellaswag", None,
           "Which ending continues the passage most plausibly?", "mit",
           lambda r, c: {"context": r["ctx"], "activity": r["activity_label"]},
           lambda r, c: list(r["endings"]),
           lambda r, c: CHOICE_IDS[int(r["label"])] if str(r["label"]) != "" else None,
           eval_split="validation", group="commonsense",
           note="long options: the 32-token option cap bites here, see the budget audit"),
    perrow("winogrande", "allenai/winogrande", "winogrande_xl",
           "Which name does the blank in the sentence refer to?", "unknown-see-card",
           lambda r, c: {"sentence": r["sentence"]},
           lambda r, c: [r["option1"], r["option2"]],
           lambda r, c: CHOICE_IDS[int(r["answer"]) - 1],
           eval_split="validation", group="commonsense"),
    perrow("mmlu", "cais/mmlu", "all",
           "Which option answers the question correctly?", "mit",
           lambda r, c: {"subject": r["subject"], "question": r["question"]},
           lambda r, c: list(r["choices"]),
           lambda r, c: CHOICE_IDS[int(r["answer"])],
           train_split="validation", group="knowledge"),
    perrow("mmlu_pro", "TIGER-Lab/MMLU-Pro", None,
           "Which option answers the question correctly?", "mit",
           lambda r, c: {"category": r["category"], "question": r["question"]},
           lambda r, c: list(r["options"]),
           lambda r, c: CHOICE_IDS[int(r["answer_index"])],
           train_split="validation", group="knowledge",
           note="up to ten options per row"),
    perrow("medmcqa", "openlifescienceai/medmcqa", None,
           "Which option answers the medical question correctly?", "apache-2.0",
           lambda r, c: {"question": r["question"]},
           lambda r, c: [r["opa"], r["opb"], r["opc"], r["opd"]],
           lambda r, c: CHOICE_IDS[int(r["cop"])],
           eval_split="validation", group="medical"),
    perrow("medqa_usmle", "GBaker/MedQA-USMLE-4-options", None,
           "Which option answers the clinical question correctly?", "cc-by-4.0",
           lambda r, c: {"vignette": r["question"]},
           lambda r, c: [r["options"][k] for k in sorted(r["options"])],
           lambda r, c: CHOICE_IDS[sorted(r["options"]).index(r["answer_idx"])],
           group="medical", note="long clinical vignettes"),
    perrow("pubmedqa", "qiaojin/PubMedQA", "pqa_labeled",
           "Does the abstract answer the research question yes, no, or maybe?", "mit",
           lambda r, c: {"question": r["question"],
                         "abstract": " ".join(r["context"]["contexts"])[:4000]},
           lambda r, c: ["yes", "no", "maybe"],
           lambda r, c: {"yes": "A", "no": "B", "maybe": "C"}.get(str(r["final_decision"]).lower()),
           eval_split="train", train_split="train", group="medical",
           note="one split upstream, so eval and fit rows are disjoint halves"),
    perrow("casehold", "coastalcph/lex_glue", "case_hold",
           "Which holding statement fits the citation in this excerpt?", "cc-by-4.0",
           lambda r, c: {"excerpt": r["context"]},
           lambda r, c: list(r["endings"]),
           lambda r, c: CHOICE_IDS[int(r["label"])],
           group="legal", note="five long holdings: the clearest test of the option cap"),
    perrow("race_middle", "ehovy/race", "middle",
           "Which option answers the question about the passage?", "other-see-card",
           lambda r, c: {"passage": r["article"], "question": r["question"]},
           lambda r, c: list(r["options"]),
           lambda r, c: str(r["answer"]).strip().upper(),
           group="reading", note="long passages"),
    perrow("truthfulqa_mc1", "truthfulqa/truthful_qa", "multiple_choice",
           "Which answer is true?", "apache-2.0",
           lambda r, c: {"question": r["question"]},
           lambda r, c: list(r["mc1_targets"]["choices"])[:12],
           lambda r, c: CHOICE_IDS[list(r["mc1_targets"]["labels"]).index(1)]
           if 1 in list(r["mc1_targets"]["labels"])[:12] else None,
           eval_split="validation", train_split="validation", group="knowledge",
           note="one split upstream, so eval and fit rows are disjoint halves"),
    perrow("cruxeval_output", "cruxeval-org/cruxeval", None,
           "Which value does the function return for the given input?", "mit",
           lambda r, c: {"code": r["code"], "input": r["input"]},
           lambda r, c: _crux_options(r, c)[0],
           lambda r, c: _crux_options(r, c)[1],
           eval_split="test", train_split="test", group="code",
           note="distractors drawn from other rows' outputs with a fixed seed; "
                "eval and fit rows are disjoint halves"),
]


# ----------------------------------------------------------------------------- per-row helpers
def _letter_gold(answer_key, labels):
    """ARC-style rows label their own options (A-D, or 1-4), so map the key through that list."""
    key = str(answer_key).strip()
    labs = [str(x).strip() for x in labels]
    if key in labs:
        return CHOICE_IDS[labs.index(key)]
    return None


def stable_hash(value):
    """A hash that is the same in every process. `hash()` is salted per run, so seeding a shuffle
    from it would quietly make the sampled rows and the option order differ between runs."""
    blob = json.dumps(value, sort_keys=True, default=str, ensure_ascii=False).encode()
    return int(hashlib.md5(blob).hexdigest()[:8], 16)


def _rowrng(ctx, row):
    """A per-row generator seeded by the run seed and the row content, so a shuffle is reproducible
    and does not depend on the order rows arrive in."""
    return random.Random(stable_hash([ctx.get("seed", 0),
                                      json.dumps(row, sort_keys=True, default=str)[:400]]))


def _shuffled(ctx, options):
    """Options in a row-stable random order, plus the id the first option ended up at.

    SciQ lists the correct answer first. Asking in that order measures position, not science.
    """
    rng = _rowrng(ctx, {"o": options})
    order = list(range(len(options)))
    rng.shuffle(order)
    return [options[i] for i in order], CHOICE_IDS[order.index(0)]


def _crux_options(row, ctx):
    """CRUXEval is a generation task upstream. Here it becomes four-way multiple choice: the real
    return value plus three return values from other rows, which keeps it answerable by a model that
    cannot emit text. The distractor pool is passed in by the loader."""
    pool = [o for o in ctx.get("crux_pool", []) if o != row["output"]]
    rng = _rowrng(ctx, {"i": row.get("id"), "c": row.get("input")})
    picks = rng.sample(pool, 3) if len(pool) >= 3 else pool
    opts, gold = _shuffled({"seed": ctx.get("seed", 0)}, [row["output"]] + list(picks))
    return opts, gold


BY_KEY = {t["key"]: t for t in TASKS}
GROUPS = sorted({t["group"] for t in TASKS})


In [ ]:
%%writefile bench_load.py
"""Turn a registry entry into typed questions plus disjoint evaluation and fitting rows.

Rules that matter for the numbers:

* **Evaluation rows and fitting rows never overlap.** Where upstream ships one split only, the split
  is permuted once with the run seed and cut in half: fitting rows from the front, evaluation rows
  from the back. A head fitted on rows it is then scored on would make the TTT column meaningless.
* **Fitting rows are sampled class by class**, round robin, so a 50-label budget on a 77-class task
  spreads over as many classes as it can reach. It still cannot cover them all, and that is the
  finding rather than a flaw in the sampling.
* **Multiple-choice rows are kept at one option count per task** (the modal count), because one
  fitted head has to span a single set of option ids. Rows off that count are dropped and counted.
"""
import random

from bench_tasks import CHOICE_IDS, stable_hash


def _load(task, split, cache_dir=None):
    """Load one split, falling back to the Hub's converted parquet branch.

    `datasets` 4 refuses to run a dataset's loading script, and several of the canonical evaluation
    sets here still ship one. For most of them the Hub holds an auto-converted parquet branch at
    `refs/convert/parquet`, which loads with the same config names and keeps the ClassLabel names,
    so trying that before giving up recovers the task rather than skipping it.
    """
    from datasets import load_dataset
    kw = {"cache_dir": cache_dir} if cache_dir else {}
    args = (task["hf"], task["config"]) if task["config"] else (task["hf"],)
    try:
        return load_dataset(*args, split=split, **kw)
    except Exception as first:
        if "script" not in str(first).lower():
            raise
        try:
            return load_dataset(*args, split=split, revision="refs/convert/parquet", **kw)
        except Exception:
            raise first


def _label_names(task, ds):
    """The option ids, in a fixed order: the registry's override, the ClassLabel names, or the
    sorted unique values of a plain string column."""
    if task.get("labels"):
        return [str(x) for x in task["labels"]]
    col = task["label_col"]
    feat = ds.features.get(col)
    names = list(getattr(feat, "names", []) or [])
    if names:
        return [str(n) for n in names]
    vals = sorted({str(v) for v in ds[col]})
    return vals


def _gold_fixed(task, row, names, class_names=()):
    """The gold option id for a fixed-option task, whatever shape upstream stores it in."""
    raw = row[task["label_col"]]
    gm = task.get("gold_map")
    if gm == "nonempty_list":
        return "true" if raw else "false"
    if isinstance(gm, dict):
        hit = gm.get(raw, gm.get(str(raw)))
        if hit is None and isinstance(raw, int) and not isinstance(raw, bool):
            for pool in (class_names, names):
                if 0 <= raw < len(pool):
                    hit = gm.get(pool[raw], gm.get(str(pool[raw])))
                if hit is not None:
                    break
        return hit
    if task["qtype"] == "score":
        return str(int(raw))
    if isinstance(raw, bool):
        return "true" if raw else "false"
    if isinstance(raw, int):
        return names[raw] if 0 <= raw < len(names) else None
    return str(raw)


def _question_fixed(task, names):
    if task["qtype"] == "boolean":
        return {"type": "boolean", "instructions": task["instructions"],
                "criteria": task.get("criteria") or {"false": "no", "true": "yes"}}
    if task["qtype"] == "score":
        return {"type": "score", "instructions": task["instructions"],
                "criteria": list(task.get("criteria") or [])}
    crit = task.get("criteria") or {}
    return {"type": "choice", "instructions": task["instructions"],
            "criteria": {n: crit.get(n) for n in names}}


def _state_fixed(task, row):
    if task.get("state_fn"):
        return task["state_fn"](row, {})
    return {c: row[c] for c in task["state_cols"] if row.get(c) is not None}


def _row_id(task, row, i):
    for key in ("id", "row_id", "uid", "pubid", "conv_id", "question_id", "unique_id", "message_id"):
        if row.get(key) not in (None, ""):
            return "%s#%s" % (task["key"], row[key])
    return "%s#%d" % (task["key"], i)


def _roundrobin(by_label, n, rng):
    """Up to n rows, one label at a time, so every class gets a turn before any gets a second."""
    keys = sorted(by_label)
    rng.shuffle(keys)
    for v in by_label.values():
        rng.shuffle(v)
    out, i = [], 0
    while len(out) < n:
        took = False
        for k in keys:
            if i < len(by_label[k]):
                out.append(by_label[k][i])
                took = True
                if len(out) >= n:
                    break
        if not took:
            break
        i += 1
    return out


def load_task(task, n_eval=60, n_fit=50, seed=20261011, cache_dir=None, max_state_chars=12000):
    """-> dict(items, fit, question(s), meta) or dict(skipped=reason)."""
    rng = random.Random(seed ^ (stable_hash(task["key"]) & 0xFFFF))
    single = task["eval_split"] == task["train_split"]
    try:
        ds_eval = _load(task, task["eval_split"], cache_dir)
        ds_fit = ds_eval if single else _load(task, task["train_split"], cache_dir)
    except Exception as exc:                        # a moved, gated or script-only dataset
        return {"skipped": "load failed: %s: %s" % (type(exc).__name__, str(exc)[:200])}

    ctx = {"seed": seed}
    if task["key"] == "cruxeval_output":            # distractor pool, drawn once per run
        ctx["crux_pool"] = list(ds_eval["output"])[:2000]

    def rows_of(ds, want, front=True, wide=False):
        """Candidate row indices: the split permuted once with the run seed, halved when upstream
        ships a single split. `wide` scans far more candidates, which is what a 151-class fit set
        needs before round robin can reach every class."""
        idx = list(range(len(ds)))
        random.Random(seed).shuffle(idx)
        if single:
            half = len(idx) // 2
            idx = idx[:half] if front else idx[half:]
        cap = min(len(idx), max(want * (120 if wide else 6), want))
        return idx[:cap]

    meta = {"licence": task["licence"], "group": task["group"], "qtype": task["qtype"],
            "hf": task["hf"], "config": task["config"], "note": task.get("note"),
            "eval_split": task["eval_split"], "fit_split": task["train_split"],
            "single_split_halved": single, "rows_upstream_eval": len(ds_eval)}

    if task["kind"] == "fixed":
        src = ds_eval if single else ds_fit
        feat = src.features.get(task["label_col"])
        class_names = [str(n) for n in (getattr(feat, "names", []) or [])]
        names = _label_names(task, src)
        question = _question_fixed(task, names)
        labels = ["false", "true"] if task["qtype"] == "boolean" else (
            [str(i) for i in range(len(task.get("criteria") or []))] if task["qtype"] == "score"
            else list(question["criteria"]))

        def collect(ds, cand, want, stratify):
            by, flat = {}, []
            for i in cand:
                row = ds[i]
                gold = _gold_fixed(task, row, names, class_names)
                if gold is None or gold not in labels:
                    continue
                state = _state_fixed(task, row)
                st = {k: (v[:max_state_chars] if isinstance(v, str) else v) for k, v in state.items()}
                item = {"id": _row_id(task, row, i), "state": st, "gold": gold}
                flat.append(item)
                by.setdefault(gold, []).append(item)
            return _roundrobin(by, want, rng) if stratify else flat[:want]

        cand_fit = rows_of(ds_fit, n_fit, front=True, wide=True)
        cand_eval = rows_of(ds_eval, n_eval, front=False)
        fit = collect(ds_fit, cand_fit, n_fit, True)
        items = collect(ds_eval, cand_eval, n_eval, False)
        fit_ids = {f["id"] for f in fit}
        items = [it for it in items if it["id"] not in fit_ids][:n_eval]
        if len(items) < min(10, n_eval):
            return {"skipped": "only %d usable evaluation rows" % len(items)}
        meta.update(n_labels=len(labels), labels_preview=labels[:12],
                    upstream_label_names=class_names[:12] or None)
        return {"kind": "fixed", "question": question, "labels": labels,
                "items": items, "fit": fit, "meta": meta}

    # ---------------------------------------------------------------- per-row options
    def build(ds, cand, want):
        out = []
        for i in cand:
            row = ds[i]
            try:
                opts = [str(o) for o in task["options_fn"](row, ctx)]
                gold = task["gold_fn"](row, ctx)
                state = task["state_fn"](row, ctx)
            except Exception:
                continue
            if not opts or gold is None or gold not in CHOICE_IDS[:len(opts)]:
                continue
            state = {k: (v[:max_state_chars] if isinstance(v, str) else v) for k, v in state.items()}
            out.append({"id": _row_id(task, row, i), "state": state, "gold": gold,
                        "options": opts})
            if len(out) >= want:
                break
        return out

    cand_fit = rows_of(ds_fit, n_fit, front=True, wide=True)
    cand_eval = rows_of(ds_eval, n_eval, front=False)
    fit_all = build(ds_fit, cand_fit, n_fit * 3)
    items_all = build(ds_eval, cand_eval, n_eval * 3)
    counts = {}
    for it in items_all:
        counts[len(it["options"])] = counts.get(len(it["options"]), 0) + 1
    if not counts:
        return {"skipped": "no usable rows after option extraction"}
    k = max(counts, key=counts.get)
    dropped = sum(v for kk, v in counts.items() if kk != k)
    items = [it for it in items_all if len(it["options"]) == k][:n_eval]
    fit_ids = set()
    fit = []
    for it in fit_all:
        if len(it["options"]) == k and it["id"] not in fit_ids:
            fit.append(it)
            fit_ids.add(it["id"])
        if len(fit) >= n_fit:
            break
    items = [it for it in items if it["id"] not in fit_ids]
    if len(items) < min(10, n_eval):
        return {"skipped": "only %d rows at the modal option count %d" % (len(items), k)}
    labels = CHOICE_IDS[:k]
    for it in items + fit:
        it["question"] = {"type": "choice", "instructions": task["instructions"],
                          "criteria": {lab: opt for lab, opt in zip(labels, it["options"])}}
    meta.update(n_labels=k, option_count=k, rows_dropped_off_modal_k=dropped,
                labels_preview=labels)
    return {"kind": "perrow", "labels": labels, "items": items, "fit": fit, "meta": meta}


In [ ]:
%%writefile bench_readouts.py
"""The readouts under test, from the frozen backbone alone up to a fitted head.

The ladder, all on the same frozen weights and the same rows:

| readout              | what answers                                     | needs labels |
|----------------------|--------------------------------------------------|--------------|
| `base_generate`      | the backbone, asked in words, greedy decode      | no           |
| `base_logprob`       | the backbone, scoring each option as text        | no           |
| `engine_no_adapter`  | the physics engine, adapters off                 | no           |
| `engine`             | the engine with its gated adapters               | no           |
| `engine_views2/3`    | the engine, averaged over surface forms          | no           |
| `ttt_k`              | a linear head on the same pooled features        | k examples   |
| `auto`               | head where it beats chance, engine elsewhere     | k examples   |

The backbone is a base checkpoint, not an instruction tune, so `base_generate` is not a straw man
of a chat model: it is the same weights the engine reads, asked the only other way they can be
asked. Its coverage number (how often the output can be parsed into an option at all) is as
interesting as its accuracy.
"""
import json
import re
import time

import numpy as np

import bench_common as bc


# ----------------------------------------------------------------------------- shared
def surface_options(q):
    """Short natural wordings of the options, for the readouts that have to put them in a prompt.

    Vega's own prompt uses `option_texts`, which carries the criterion as well. A base model asked
    to pick among "false - no, the statement does not hold" style strings is being handicapped by
    the harness rather than measured, so the baselines get the plain wording.
    """
    t, crit = q["type"], q.get("criteria")
    if t == "boolean":
        return ["no", "yes"]
    if t == "score":
        return [str(c) for c in (crit or [])]
    return [str(v) if v else str(k) for k, v in (crit or {}).items()]


def _state_text(state):
    return bc.serialize_state(state) if not isinstance(state, str) else state


def normalise_answer(a, qtype, labels):
    """One Vega answer in the harness's own vocabulary: probabilities keyed by option id.

    Needed because the engine reports a boolean as `p(true)` with `probs` keyed "False"/"True",
    a score with integer keys, and a choice with the criteria's own keys. Scoring code that reads
    those three shapes inline is where silent off-by-one label bugs live.
    """
    if qtype == "boolean":
        p = float(a.get("boolean", 0.0))
        probs = {"false": 1.0 - p, "true": p}
    else:
        probs = {str(k): float(v) for k, v in (a.get("probs") or {}).items()}
    probs = {k: probs.get(k, 0.0) for k in labels}
    pred = max(probs, key=probs.get) if probs else None
    pset = a.get("prediction_set")
    ids = None
    if pset is not None:
        ids = [str(x).lower() if qtype == "boolean" else str(x) for x in pset]
    return probs, pred, ids


# ----------------------------------------------------------------------------- Vega
class VegaRunner:
    """One loaded Vega: the engine readouts, the pooled features, and the fitted heads."""

    def __init__(self, size, device, adapters="auto", max_len=None, token=None):
        import vegaml

        self.size = size
        self.device = device
        self.vega = vegaml.load(size, device=device, adapters=adapters, mode="engine",
                                max_len=max_len, token=token)
        self.tok = self.vega.engine.perc.tok
        self.cfg = self.vega.engine.cfg
        self.adapter_names = [a["name"] for a in (self.vega.engine.adapters or [])]

    def set_adapters(self, mode):
        self.vega.engine.set_adapters(mode)

    def free(self):
        """Drop the weights, keeping the metadata later cells read.

        `cfg`, `tok` and `adapter_names` stay, because the base phase asks this object for the
        backbone's name after the engine itself is gone. The cycle collector has to run before
        emptying the CUDA cache or nothing is actually returned.
        """
        import gc

        self.vega = None
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass

    def engine_record(self, item, question, labels, views=1, alpha="0.1"):
        t0 = time.perf_counter()
        out = self.vega.decide(item["state"], {"q": question}, mode="engine", alpha=alpha,
                               views=views)
        dt = (time.perf_counter() - t0) * 1000
        a = (out.get("answers") or {}).get("q") or {}
        probs, pred, ids = normalise_answer(a, question["type"], labels)
        rec = {"id": item["id"], "gold": item["gold"], "pred": pred, "probs": probs,
               "latency_ms": dt, "abstain": bool(a.get("abstain")),
               "unbound": bool(a.get("unbound")), "n_tokens": out.get("n_tokens"),
               "settle_steps": a.get("settle_steps"), "gate": a.get("gate"),
               "adapter": a.get("adapter")}
        if ids is not None:
            rec["set_size"] = len(ids)
            rec["set_hit"] = item["gold"] in ids
        return rec

    def feature(self, item, question):
        return self.vega.features(item["state"], question)

    def features_for(self, items, question=None):
        """Pooled features for a list of items, one read each. `question=None` means per-row."""
        return [self.feature(it, question or it["question"]) for it in items]


def fit_head_ladder(X, y, labels, shots, folds=5):
    """One head per budget in `shots`, each fitted on the first k examples of the same feature set.

    Fitting on prefixes costs nothing extra: the features are computed once. What it buys is a
    learning curve, which is the only honest way to answer "is a head worth it", since the answer
    is a function of how many labels you have.
    """
    from vegaml import TTTHead

    out = {}
    for k in shots:
        if k > len(X):
            continue
        Xk, yk = X[:k], y[:k]
        if len(set(yk)) < 2:
            out[k] = {"fitted": False, "reason": "one class only in the first %d examples" % k}
            continue
        head = TTTHead(labels)
        classes = len(set(yk))
        # Folds are stratified, so a class needs one example per fold to survive the split. Five
        # folds over fifty examples of six classes is fine; five folds over six examples is not,
        # and asking for them returns None, which "auto" then reads as an unmeasured head and
        # leaves to the engine. Scaling the fold count to what the data can support is the
        # difference between a measured head and an unknown one at the small budgets.
        folds_k = max(2, min(folds, len(yk) // max(classes, 1)))
        try:
            head.fit(Xk, yk)
            cv = head.cross_val_accuracy(Xk, yk, folds=folds_k)
        except Exception as exc:
            out[k] = {"fitted": False, "reason": "%s: %s" % (type(exc).__name__, str(exc)[:160])}
            continue
        chance = 1.0 / max(len(labels), 2)
        out[k] = {"fitted": True, "head": head, "cv_accuracy": cv, "chance": round(chance, 4),
                  "cv_n": getattr(head, "cross_val_n", 0), "classes_seen": classes,
                  "folds": folds_k, "examples": len(Xk),
                  "beats_chance": None if cv is None else bool(cv > chance)}
        if cv is None:
            out[k]["cv_note"] = ("%d examples over %d classes is too few to cross-validate, so "
                                 "this head's accuracy is unknown rather than good or bad, and "
                                 "auto leaves the question to the engine" % (len(yk), classes))
    return out


def head_record(head, feat, item, labels, allowed=None):
    t0 = time.perf_counter()
    p = head.predict(feat, allowed=allowed or labels)
    dt = (time.perf_counter() - t0) * 1000
    tot = sum(p.values()) or 1.0
    probs = {k: v / tot for k, v in p.items()}
    return {"id": item["id"], "gold": item["gold"], "pred": max(probs, key=probs.get),
            "probs": probs, "latency_ms": dt}


def auto_record(engine_rec, head_rec, head_info):
    """What mode="auto" would answer: the head when its cross-validated accuracy beat chance on
    this task, the engine otherwise. Reported as its own column so the routing rule is visible."""
    use_head = bool(head_info and head_info.get("beats_chance"))
    rec = dict(head_rec if use_head else engine_rec)
    rec["readout"] = "ttt" if use_head else "engine"
    return rec


# ----------------------------------------------------------------------------- base backbone
class BaseLM:
    """The frozen backbone on its own: asked in words, or scoring each option as a continuation.

    Loaded as a second copy of the weights because Vega's perception registers forward hooks that
    stop the forward pass at the deepest layer it reads. Those hooks make the shared instance
    unusable for generation or for logits, which is why this class does not borrow it.
    """

    def __init__(self, repo, device, dtype=None, max_len=3072, batch=16, token=None,
                 thinking=False):
        import gc

        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self.torch = torch
        self.device = torch.device(device)
        if dtype is None:
            dtype = torch.float16 if self.device.type == "cuda" else torch.float32
        self.tok = AutoTokenizer.from_pretrained(repo, token=token)
        if self.tok.pad_token_id is None:
            self.tok.pad_token = self.tok.eos_token
        self.mem_before = bc.memory_gb()
        # The 4B is about 8 GB of fp16 weights, and `from_pretrained(...).to(device)` materialises
        # all of it in host memory first. A second copy of that, on top of two resident Vega
        # instances, is enough to have the kernel killed outright, which arrives as a dead kernel
        # with no traceback. device_map streams the shards straight to the card instead, so the
        # host only ever holds one shard. The plain path stays as a fallback for an image without
        # accelerate, and which path ran is recorded.
        self.model, self.load_mode, failures = None, None, []
        for mode in ("device_map", "plain"):
            kw = dict(dtype=dtype, attn_implementation="sdpa", token=token)
            if mode == "device_map":
                kw["device_map"] = {"": str(self.device)}
            try:
                self.model = AutoModelForCausalLM.from_pretrained(repo, **kw)
                if mode == "plain":
                    self.model.to(self.device)
                self.load_mode = mode
                break
            except Exception as exc:
                failures.append("%s: %s: %s" % (mode, type(exc).__name__, str(exc)[:160]))
                self.model = None
                gc.collect()
                if self.device.type == "cuda":
                    torch.cuda.empty_cache()
        if self.model is None:
            raise RuntimeError("could not load %s; tried %s" % (repo, " | ".join(failures)))
        self.load_failures = failures
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.mem_after = bc.memory_gb()
        self.max_len = max_len
        self.batch = batch
        self.repo = repo
        self.thinking = thinking
        self.template_takes_thinking = "enable_thinking" in str(
            getattr(self.tok, "chat_template", "") or "")

    # ---------------------------------------------------------------- prompting
    def _fit_state(self, state, reserve):
        """The state, cut to whatever the prompt budget leaves. Cut rather than refused, because a
        baseline that declines to read long rows would look better than it is; the cut is recorded."""
        text = _state_text(state)
        ids = self.tok(text, add_special_tokens=False)["input_ids"]
        room = max(64, self.max_len - reserve)
        if len(ids) <= room:
            return text, False
        return self.tok.decode(ids[:room]), True

    def _prompt(self, state, question, letters):
        """The prompt, whether the state had to be cut, and whether a reasoning block is left open.

        Qwen3.5's chat template opens a `<think>` block, and the two sizes of the same family
        default the other way round: with `enable_thinking` unset the 0.8B template closes the
        block inside the generation prompt and the 4B leaves it open. Left open, a short generation
        budget is spent reasoning and the baseline scores near zero for a formatting reason rather
        than a modelling one, so thinking is switched off explicitly and the same prompt shape
        reaches both sizes. The flag comes back too, so a run with thinking on stays readable
        instead of quietly wrong.
        """
        opts = surface_options(question)
        lines = ["%s) %s" % (l, o) for l, o in zip(letters, opts)]
        head = ("You are a decision model. Read the record and answer the question with exactly one "
                "option id.\n\n")
        tail = ("\n\nQuestion: %s\nOptions:\n%s\n\nAnswer with one id from %s and nothing else.\n"
                "Answer:" % (question["instructions"], "\n".join(lines),
                             ", ".join(letters)))
        reserve = len(self.tok(head + tail, add_special_tokens=False)["input_ids"]) + 16
        body, cut = self._fit_state(state, reserve)
        text = head + "Record:\n" + body + tail
        if getattr(self.tok, "chat_template", None):
            kw = {"tokenize": False, "add_generation_prompt": True}
            if self.template_takes_thinking:
                kw["enable_thinking"] = bool(self.thinking)
            text = self.tok.apply_chat_template([{"role": "user", "content": text}], **kw)
        think_open = text.count("<think>") > text.count("</think>")
        return text, cut, think_open

    # ---------------------------------------------------------------- readouts
    def generate_record(self, item, question, labels, max_new_tokens=12):
        letters = [chr(ord("A") + i) for i in range(len(labels))]
        text, cut, think_open = self._prompt(item["state"], question, letters)
        ids = self.tok(text, return_tensors="pt").to(self.device)
        t0 = time.perf_counter()
        with self.torch.inference_mode():
            out = self.model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False,
                                      pad_token_id=self.tok.pad_token_id)
        dt = (time.perf_counter() - t0) * 1000
        gen = self.tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)
        answer = gen
        if think_open:
            # the answer is whatever follows the close tag. Reasoning that never closed is not an
            # answer, and reading an option out of the reasoning would invent a result: 17 of the
            # 43 tasks in the first full run scored above zero that way, from text that was the
            # model thinking aloud rather than answering.
            answer = gen.split("</think>", 1)[1] if "</think>" in gen else ""
        pred = parse_generation(answer, question, labels, letters)
        return {"id": item["id"], "gold": item["gold"], "pred": pred, "probs": None,
                "latency_ms": dt, "raw_len": len(gen), "answer_len": len(answer),
                "thinking_open": think_open, "state_truncated": cut, "parsed": pred is not None}

    def logprob_record(self, item, question, labels):
        """Each option scored as the continuation of the same prompt.

        Two numbers come out of one batch: the mean log probability per option token (what the
        `base_logprob` column reports, because it does not reward short options) and the summed log
        probability (`base_logprob_sum`), which does. Reporting both keeps the normalisation choice
        from quietly deciding the result.
        """
        torch = self.torch
        letters = [chr(ord("A") + i) for i in range(len(labels))]
        prompt, cut, _ = self._prompt(item["state"], question, letters)
        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        opts = surface_options(question)
        conts = [self.tok(" " + o, add_special_tokens=False)["input_ids"][:48] or
                 self.tok(" ?", add_special_tokens=False)["input_ids"] for o in opts]
        sums, means = [], []
        t0 = time.perf_counter()
        for start in range(0, len(conts), self.batch):
            chunk = conts[start:start + self.batch]
            seqs = [p_ids + c for c in chunk]
            width = max(len(s) for s in seqs)
            pad = self.tok.pad_token_id or 0
            inp = torch.full((len(seqs), width), pad, dtype=torch.long)
            att = torch.zeros((len(seqs), width), dtype=torch.long)
            for i, s in enumerate(seqs):                 # left padding keeps the tail aligned
                inp[i, width - len(s):] = torch.tensor(s)
                att[i, width - len(s):] = 1
            inp, att = inp.to(self.device), att.to(self.device)
            with torch.inference_mode():
                logits = self.model(input_ids=inp, attention_mask=att).logits.float()
            logp = torch.log_softmax(logits, dim=-1)
            for i, c in enumerate(chunk):
                n = len(c)
                # the token at position t is predicted by the logits at t-1
                pos = torch.arange(width - n - 1, width - 1, device=self.device)
                tgt = torch.tensor(c, device=self.device)
                lp = logp[i, pos, tgt]
                sums.append(float(lp.sum()))
                means.append(float(lp.mean()))
        dt = (time.perf_counter() - t0) * 1000

        def to_probs(scores):
            z = np.array(scores, dtype=np.float64)
            z = z - z.max()
            e = np.exp(z)
            return {lab: float(v) for lab, v in zip(labels, e / e.sum())}

        pm, ps = to_probs(means), to_probs(sums)
        return ({"id": item["id"], "gold": item["gold"], "pred": max(pm, key=pm.get), "probs": pm,
                 "latency_ms": dt, "state_truncated": cut},
                {"id": item["id"], "gold": item["gold"], "pred": max(ps, key=ps.get), "probs": ps,
                 "latency_ms": dt, "state_truncated": cut})

    def free(self):
        """Drop the weights and hand the memory back, host side as well as device side.

        `del self.model` alone is not enough: transformers modules sit in reference cycles, so the
        memory is only reclaimed once the cycle collector has run, and emptying the CUDA cache
        before that frees nothing. Calling this twice is safe.
        """
        import gc

        self.model = None
        gc.collect()
        if self.device.type == "cuda":
            self.torch.cuda.empty_cache()


_YES = re.compile(r"\b(yes|true|y)\b", re.I)
_NO = re.compile(r"\b(no|false|n)\b", re.I)


def parse_generation(text, question, labels, letters):
    """The option a generated answer names, or None when it names none.

    None is not the same as wrong: it lowers `coverage`, and that is the number that says whether a
    base model asked in words can be put in a pipeline at all.
    """
    t = (text or "").strip()
    if not t:
        return None
    if question["type"] == "boolean":
        head = t[:40]
        y, n = _YES.search(head), _NO.search(head)
        if y and (not n or y.start() < n.start()):
            return "true"
        if n:
            return "false"
    m = re.match(r"\s*\(?([A-Z])\)?[\s.:,)]", t + " ")
    if m and m.group(1) in letters:
        return labels[letters.index(m.group(1))]
    for lab, letter, surf in zip(labels, letters, surface_options(question)):
        if re.search(r"\b%s\b" % re.escape(str(lab)), t, re.I):
            return lab
        if surf and re.search(r"\b%s\b" % re.escape(str(surf)[:40]), t, re.I):
            return lab
    if question["type"] == "score":
        d = re.search(r"\b(\d+)\b", t)
        if d and d.group(1) in labels:
            return d.group(1)
    return None


In [ ]:
%%writefile bench_jev.py
"""The hosted reference: Jev's typed-decision endpoint, called with the same questions.

Jev takes the same request shape Vega answers locally, so the comparison needs no re-framing: one
state, a map of typed questions, a probability per declared option. Two differences are handled
here and nowhere else, so the rest of the harness stays uniform:

* Jev calls a yes/no question `noul`; Vega calls it `boolean`.
* Jev answers over the network, so a row can fail. A failed row is recorded as unanswered rather
  than retried into a different number, and it lowers `coverage` like any other refusal.

Responses are cached on disk by request hash. A re-run of the notebook therefore costs nothing and
returns the same answers, which also means the Jev column is reproducible within a session.
"""
import hashlib
import json
import os
import random
import threading
import time
import urllib.error
import urllib.request

ENDPOINT = "https://api.typesafe.ai/v1/systemone"


def to_jev_questions(questions):
    """Vega's question map in Jev's vocabulary."""
    out = {}
    for qid, q in questions.items():
        jq = {"instructions": q["instructions"]}
        if q["type"] == "boolean":
            jq["type"] = "noul"
            jq["criteria"] = q.get("criteria") or {"false": "no", "true": "yes"}
        elif q["type"] == "score":
            jq["type"] = "score"
            jq["criteria"] = list(q.get("criteria") or [])
        else:
            jq["type"] = "choice"
            jq["criteria"] = {k: (v or k) for k, v in q["criteria"].items()}
        out[qid] = jq
    return out


def parse_answer(entry, qtype, labels):
    """One Jev answer as (probs, pred). Returns (None, None) when the shape is not understood,
    which is counted rather than guessed at."""
    if not isinstance(entry, dict):
        return None, None
    probs = None
    for key in ("probabilities", "probs", "distribution"):
        if isinstance(entry.get(key), dict):
            probs = {str(k): float(v) for k, v in entry[key].items()}
            break
    if probs is None and qtype == "boolean":
        for key in ("noul", "boolean", "probability", "p_true"):
            if isinstance(entry.get(key), (int, float)):
                p = float(entry[key])
                probs = {"false": 1.0 - p, "true": p}
                break
    if probs is None and isinstance(entry.get("scores"), dict):
        probs = {str(k): float(v) for k, v in entry["scores"].items()}
    if probs is not None:
        low = {k.lower(): v for k, v in probs.items()}
        picked = {}
        for lab in labels:
            if lab in probs:
                picked[lab] = probs[lab]
            elif str(lab).lower() in low:
                picked[lab] = low[str(lab).lower()]
        if picked and sum(picked.values()) > 0:
            tot = sum(picked.values())
            picked = {k: v / tot for k, v in picked.items()}
            return picked, max(picked, key=picked.get)
    # label-only fallback: no distribution, so calibration is not measurable for this row
    for key in ("choice", "score", "label", "answer", "noul"):
        if key in entry:
            val = entry[key]
            if qtype == "boolean" and isinstance(val, (int, float)):
                return None, ("true" if float(val) >= 0.5 else "false")
            sval = str(val)
            if sval in labels:
                return None, sval
            low = {str(l).lower(): l for l in labels}
            if sval.lower() in low:
                return None, low[sval.lower()]
    return None, None


class JevClient:
    def __init__(self, api_key, model="jev-latest", endpoint=ENDPOINT, cache_path=None,
                 timeout=60, retries=4, max_rps=None):
        if not (api_key or "").strip():
            raise ValueError("a Jev API key is required; the notebook reads it from the "
                             "JEV_API Kaggle secret")
        self._key = api_key.strip()
        self.model, self.endpoint, self.timeout, self.retries = model, endpoint, timeout, retries
        self.cache_path = cache_path
        self.cache = {}
        self.lock = threading.Lock()
        self.stats = {"calls": 0, "cache_hits": 0, "http_errors": {}, "usage": {},
                      "decisions": 0, "latency_ms": []}
        self._min_gap = (1.0 / max_rps) if max_rps else 0.0
        self._last = 0.0
        if cache_path and os.path.exists(cache_path):
            with open(cache_path) as fh:
                for line in fh:
                    try:
                        rec = json.loads(line)
                        self.cache[rec["k"]] = rec["v"]
                    except Exception:
                        continue

    # ---------------------------------------------------------------- transport
    def _body(self, state, questions):
        return {"model": self.model, "state": state, "questions": to_jev_questions(questions)}

    def _key_for(self, body):
        blob = json.dumps(body, sort_keys=True, ensure_ascii=False)
        return hashlib.sha256(blob.encode()).hexdigest()

    def _remember(self, key, value):
        with self.lock:
            self.cache[key] = value
            if self.cache_path:
                with open(self.cache_path, "a") as fh:
                    fh.write(json.dumps({"k": key, "v": value}) + "\n")

    def _throttle(self):
        if not self._min_gap:
            return
        with self.lock:
            wait = self._last + self._min_gap - time.time()
            if wait > 0:
                time.sleep(wait)
            self._last = time.time()

    def raw(self, state, questions):
        """The decoded response, from cache when it is there. -> (payload, latency_ms, error)."""
        body = self._body(state, questions)
        key = self._key_for(body)
        if key in self.cache:
            with self.lock:
                self.stats["cache_hits"] += 1
            return self.cache[key], 0.0, None
        data = json.dumps(body).encode()
        last = "no attempt made"
        for attempt in range(self.retries + 1):
            self._throttle()
            req = urllib.request.Request(
                self.endpoint, data=data, method="POST",
                headers={"Authorization": "Bearer " + self._key,
                         "Content-Type": "application/json",
                         "User-Agent": "vega-bench/1.0"})
            t0 = time.perf_counter()
            try:
                with urllib.request.urlopen(req, timeout=self.timeout) as resp:
                    payload = json.loads(resp.read())
                dt = (time.perf_counter() - t0) * 1000
                with self.lock:
                    self.stats["calls"] += 1
                    self.stats["latency_ms"].append(dt)
                    for k, v in (payload.get("usage") or {}).items():
                        if isinstance(v, (int, float)):
                            self.stats["usage"][k] = self.stats["usage"].get(k, 0) + v
                self._remember(key, payload)
                return payload, dt, None
            except urllib.error.HTTPError as exc:
                detail = exc.read().decode("utf-8", "ignore")[:200]
                last = "HTTP %d: %s" % (exc.code, detail)
                with self.lock:
                    self.stats["http_errors"][str(exc.code)] = \
                        self.stats["http_errors"].get(str(exc.code), 0) + 1
                if exc.code in (400, 401, 403, 404, 422):     # a retry cannot fix the request
                    return None, (time.perf_counter() - t0) * 1000, last
            except Exception as exc:
                last = "%s: %s" % (type(exc).__name__, str(exc)[:160])
            time.sleep(min(20.0, (2 ** attempt) + random.random()))
        return None, 0.0, last

    # ---------------------------------------------------------------- one row
    def record(self, item, question, labels):
        payload, dt, err = self.raw(item["state"], {"q": question})
        rec = {"id": item["id"], "gold": item["gold"], "pred": None, "probs": None,
               "latency_ms": dt or None}
        if err or not isinstance(payload, dict):
            rec["error"] = err or "unparsable response"
            return rec
        answers = payload.get("answers") or {}
        entry = answers.get("q") or (list(answers.values())[0] if answers else None)
        probs, pred = parse_answer(entry, question["type"], labels)
        rec["probs"], rec["pred"] = probs, pred
        if pred is None:
            rec["error"] = "answer shape not understood: %s" % json.dumps(entry)[:160]
        with self.lock:
            self.stats["decisions"] += 1
        rec["model_reported"] = payload.get("model")
        return rec

    def run(self, items, question_for, labels_for, workers=6, budget=None, progress=None):
        """Every item, in a small thread pool. `question_for(item)` and `labels_for(item)` keep
        per-row option sets working. Stops early and returns what it has if `budget` runs out."""
        from concurrent.futures import ThreadPoolExecutor

        out = []
        with ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {}
            for it in items:
                futures[pool.submit(self.record, it, question_for(it), labels_for(it))] = it
            for i, fut in enumerate(futures):
                if budget is not None and not budget.ok(5):
                    break
                try:
                    out.append(fut.result())
                except Exception as exc:
                    out.append({"id": futures[fut]["id"], "gold": futures[fut]["gold"],
                                "pred": None, "probs": None,
                                "error": "%s: %s" % (type(exc).__name__, str(exc)[:120])})
                if progress and i % 50 == 0:
                    progress(i + 1, len(items))
        return out

    def summary(self):
        import numpy as np
        lat = self.stats["latency_ms"]
        return {"model": self.model, "endpoint": self.endpoint, "calls": self.stats["calls"],
                "cache_hits": self.stats["cache_hits"], "decisions": self.stats["decisions"],
                "http_errors": dict(self.stats["http_errors"]),
                "usage_totals": dict(self.stats["usage"]),
                "latency_ms_p50": round(float(np.percentile(lat, 50)), 1) if lat else None,
                "latency_ms_p95": round(float(np.percentile(lat, 95)), 1) if lat else None}


In [ ]:
%%writefile bench_suites.py
"""Two third-party typed-decision suites, loaded at run time and never redistributed.

* **JevBench** public tiers (MIT, `fstandhartinger/jevbench`): already written as typed decisions,
  with paraphrase groups, so it also measures whether the same rule in different wordings gets the
  same answer. The published leaderboard uses these plus held-out items, so our numbers on the
  public tiers are comparable in kind, not identical in value.
* **DecisionBench** (`Hanno-Labs/decision-bench`): 43 tasks over three primitives, with gold
  probability distributions as well as gold labels.

Neither is used to fit anything. They are other people's evaluation sets held under their own
licences, and the one thing that would spoil them is training on them, so nothing here does: no
head is fitted on these rows, and the saved output holds row ids and metrics only.
"""
import json
import random
import urllib.request

JEVBENCH_RAW = ("https://raw.githubusercontent.com/fstandhartinger/jevbench/main/"
                "datasets/public/%s.jsonl")
JEV_TIERS = ("original", "easy", "hard")


# ----------------------------------------------------------------------------- JevBench
def fetch_jevbench(tiers=JEV_TIERS, timeout=60):
    """-> (items, meta). An unreachable tier is reported, not silently dropped."""
    items, errors = [], {}
    for tier in tiers:
        try:
            with urllib.request.urlopen(JEVBENCH_RAW % tier, timeout=timeout) as resp:
                body = resp.read().decode("utf-8")
        except Exception as exc:
            errors[tier] = "%s: %s" % (type(exc).__name__, str(exc)[:160])
            continue
        for line in body.splitlines():
            line = line.strip()
            if not line:
                continue
            try:
                raw = json.loads(line)
            except Exception:
                continue
            conv = _jev_item(raw, tier)
            if conv:
                items.append(conv)
    return items, {"source": "fstandhartinger/jevbench public tiers", "licence": "mit",
                   "tiers": list(tiers), "errors": errors, "n": len(items)}


def _jev_item(raw, tier):
    q = raw.get("question") or {}
    t = {"noul": "boolean", "choice": "choice", "score": "score"}.get(q.get("type"))
    if not t or "state" not in raw:
        return None
    question = {"type": t, "instructions": q.get("instructions") or "", "criteria": q.get("criteria")}
    expected = raw.get("expected")
    if t == "boolean":
        gold = {"yes": "true", "true": "true", "no": "false", "false": "false"}.get(
            str(expected).strip().lower())
        labels = ["false", "true"]
        if not question["criteria"]:
            question["criteria"] = {"false": "no", "true": "yes"}
    elif t == "score":
        crit = list(question["criteria"] or [])
        labels = [str(i) for i in range(len(crit))]
        s = str(expected).strip()
        gold = s if s in labels else (str(crit.index(expected)) if expected in crit else None)
    else:
        crit = question["criteria"] or {}
        labels = [str(k) for k in crit]
        gold = str(expected) if str(expected) in labels else None
    if gold is None:
        return None
    return {"id": "jevbench:%s:%s" % (tier, raw.get("id")), "state": raw["state"], "gold": gold,
            "question": question, "labels": labels, "tier": tier,
            "family": raw.get("family"), "group": raw.get("group"), "qtype": t}


def paraphrase_consistency(items, records):
    """The share of paraphrase groups answered the same way throughout.

    A group is one rule written several ways. Scoring only accuracy hides a model that is right on
    one wording and wrong on its twin, which in a pipeline is the same thing as being unreliable.
    """
    by_id = {r["id"]: r for r in records}
    groups = {}
    for it in items:
        if it.get("group"):
            groups.setdefault(it["group"], []).append(it)
    whole, consistent, pairs_same = 0, 0, []
    for members in groups.values():
        preds = [by_id.get(m["id"], {}).get("pred") for m in members if m["id"] in by_id]
        if len(preds) < 2:
            continue
        whole += 1
        consistent += int(len(set(preds)) == 1)
        pairs_same.append(len(set(preds)) == 1)
    if not whole:
        return {}
    return {"paraphrase_groups": whole,
            "paraphrase_consistency": round(consistent / whole, 4)}


# ----------------------------------------------------------------------------- DecisionBench
# The affirmative and negative sides of a two-candidate row, matched on the first word of the
# label. Measured against the real label pairs in the suite: Yes/No and Relevant/Not relevant
# account for most of them, "Yes - the evidence is sufficient" style labels are caught by the first
# word, and genuinely non-polar pairs such as Retain verbatim / Summarize fall through on purpose
# and are asked as a two-option choice instead.
_POS = {"yes", "true", "relevant", "safe", "permitted", "permit", "allowed", "allow", "approve",
        "approved", "accept", "accepted", "retain", "valid", "compliant", "sufficient"}
_NEG = {"no", "false", "irrelevant", "unsafe", "prohibited", "prohibit", "denied", "deny",
        "reject", "rejected", "discard", "invalid", "noncompliant", "insufficient", "block",
        "blocked"}


def fetch_decisionbench(per_task=30, seed=20261011, cache_dir=None, max_state_chars=12000,
                        binary_as_boolean=True):
    """-> (items, meta). `per_task` rows per task_id, sampled with a fixed seed."""
    from datasets import load_dataset

    kw = {"cache_dir": cache_dir} if cache_dir else {}
    ds = load_dataset("Hanno-Labs/decision-bench", split="eval", **kw)
    by_task = {}
    for i, task in enumerate(ds["task_id"]):
        by_task.setdefault(task, []).append(i)
    rng = random.Random(seed)
    picked = []
    for task in sorted(by_task):
        idx = by_task[task]
        rng.shuffle(idx)
        picked.extend(idx[:per_task])
    items, skipped, as_bool = [], 0, 0
    primitives = {}
    for i in sorted(picked):
        row = ds[i]
        try:
            cands = json.loads(row["candidates_json"])
            state = json.loads(row["state_json"])
        except Exception:
            skipped += 1
            continue
        if not isinstance(cands, list) or len(cands) < 2:
            skipped += 1
            continue
        ids = [str(c.get("id")) for c in cands]
        labs = [str(c.get("label") or c.get("id")) for c in cands]
        descs = [c.get("description") for c in cands]
        gold_i = ids.index(str(row["gold_candidate_id"])) if str(row["gold_candidate_id"]) in ids \
            else None
        if gold_i is None:
            skipped += 1
            continue
        prim = row["primitive"]
        primitives[prim] = primitives.get(prim, 0) + 1
        soft = list(row.get("gold_probabilities") or [])
        text = json.dumps(state, ensure_ascii=False)[:max_state_chars]
        if prim.startswith("binary") and binary_as_boolean and _polarity(labs) is not None:
            pos = _polarity(labs)
            question = {"type": "boolean", "instructions": row["instruction"],
                        "criteria": {"true": labs[pos], "false": labs[1 - pos]}}
            labels = ["false", "true"]
            gold = "true" if gold_i == pos else "false"
            soft = [soft[1 - pos], soft[pos]] if len(soft) == 2 else []
            as_bool += 1
        else:
            crit = {}
            for lab, desc, cid in zip(labs, descs, ids):
                key = lab if lab not in crit else "%s (%s)" % (lab, cid)
                crit[key] = desc
            question = {"type": "choice", "instructions": row["instruction"], "criteria": crit}
            labels = list(crit)
            gold = labels[gold_i]
        items.append({"id": "decisionbench:%s" % row["row_id"], "state": text, "gold": gold,
                      "question": question, "labels": labels, "qtype": question["type"],
                      "task": row["task_id"], "primitive": prim, "family": row.get("family"),
                      "domain": row.get("domain"), "candidate_count": len(cands),
                      "reasoning_required": bool(row.get("reasoning_required")),
                      "soft_gold": [float(x) for x in soft] if len(soft) == len(labels) else None})
    return items, {"source": "Hanno-Labs/decision-bench", "licence": "other-see-card",
                   "ordinal_rows_asked_as_choice": True,
                   "ordinal_note": ("ordinal_scoring rows are asked as a choice, not as a score. "
                                    "Their candidate lists are shuffled per row and the stored "
                                    "ordinal_value follows that position rather than the severity "
                                    "of the label, so one row runs Negligible to Critical, the "
                                    "next runs Critical to Negligible, and a third interleaves "
                                    "them. A score-typed question asserts that level i is below "
                                    "level i+1, which these rows do not carry, so asking them as "
                                    "a score would score the harness and not the model. The score "
                                    "type is exercised on JevBench's ordinal family and on the "
                                    "ordinal tasks in the public set instead."),
                   "split": "eval", "per_task": per_task, "tasks": len(by_task),
                   "rows_upstream": len(ds), "n": len(items), "rows_skipped": skipped,
                   "binary_rows_asked_as_boolean": as_bool,
                   "primitives_seen": primitives,
                   "binary_note": ("a two-candidate row is asked as a yes/no only when one of its "
                                   "labels is clearly the affirmative side. Pairs that are not "
                                   "polar, such as Retain verbatim against Summarize, are asked "
                                   "as a two-option choice, because guessing the polarity would "
                                   "invert the gold answer for that row.")}


def _first_word(label):
    import re
    m = re.search(r"[A-Za-z]+", str(label))
    return m.group(0).lower() if m else ""


def _polarity(labels):
    """Which of two candidate labels is the affirmative one, or None when neither clearly is.

    None is the safe answer: the row is then asked as a two-option choice, which decides nothing
    about polarity. Guessing wrong would invert the gold for that row.
    """
    if len(labels) != 2:
        return None
    low = [str(x).strip().lower() for x in labels]
    if low[0] == low[1]:
        return None
    for i in (0, 1):                              # "Relevant" against "Not relevant"
        other = low[1 - i]
        if other in ("not " + low[i], "non-" + low[i], "no " + low[i], low[i] + " not"):
            return i
    heads = [_first_word(x) for x in labels]
    for i in (0, 1):
        if heads[i] in _POS and heads[1 - i] in _NEG:
            return i
    return None


def soft_gold_kl(items, records):
    """Mean KL(gold || predicted) over rows whose gold is a distribution rather than a single label.

    DecisionBench ships gold probabilities, so on the rows where the gold is genuinely uncertain a
    model that commits hard is wrong in a way accuracy cannot see.
    """
    import math

    by_id = {r["id"]: r for r in records}
    kls, n_soft = [], 0
    for it in items:
        soft = it.get("soft_gold")
        rec = by_id.get(it["id"])
        if not soft or not rec or not rec.get("probs"):
            continue
        if max(soft) >= 0.999:            # one-hot gold: KL would just restate the NLL
            continue
        n_soft += 1
        kl = 0.0
        for lab, g in zip(it["labels"], soft):
            if g <= 0:
                continue
            p = max(float(rec["probs"].get(lab, 0.0)), 1e-12)
            kl += g * math.log(g / p)
        kls.append(kl)
    if not kls:
        return {"soft_gold_rows": n_soft}
    return {"soft_gold_rows": n_soft,
            "soft_gold_kl_mean": round(float(sum(kls) / len(kls)), 4)}


In [ ]:
%%writefile bench_run.py
"""Orchestration: run the readouts over a suite, collect metrics, save after every task.

The result file is written incrementally. Kaggle stops a committed notebook at twelve hours and the
4B is the slow part, so a run that is cut short still leaves every task it finished, with a
`completed` flag saying so. Nothing here deletes or rewrites earlier phases, so the notebook can
call the phases cell by cell and watch them land.
"""
import gc
import json
import os
import time

import bench_common as bc
import bench_readouts as RO


class Results:
    def __init__(self, cfg, path, resume_from=None):
        self.path = path
        self.data = {"schema": 1, "created": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                     "config": cfg, "environment": {}, "suites": {}, "results": {}, "phases": {},
                     "notes": [], "completed": False}
        if resume_from:
            self.resume(resume_from)

    def resume(self, src):
        """Carry a previous run's finished readouts forward so they are not paid for twice.

        A kernel that dies loses the session but not the file: the output JSON is written after
        every task. Attach the previous version's output as an input dataset, point
        `CFG["resume_from"]` at it, and every readout already in it is skipped. Nothing is
        recomputed and nothing already measured is overwritten, so the two runs' numbers sit in
        one file exactly as they were measured.
        """
        if not os.path.exists(src):
            self.note("resume_from %s does not exist, starting from nothing" % src)
            return
        with open(src) as fh:
            old = json.load(fh)
        self.data["results"] = old.get("results") or {}
        self.data["suites"] = old.get("suites") or {}
        self.data["phases"] = {("prior_" + k): v for k, v in (old.get("phases") or {}).items()}
        self.data["notes"] = ["prior run: " + n for n in (old.get("notes") or [])]
        self.data["resumed_from"] = {
            "path": src, "created": old.get("created"), "completed": old.get("completed"),
            "wall_clock_minutes": old.get("wall_clock_minutes"),
            "environment": old.get("environment"), "config": old.get("config")}
        self.note("resumed from %s: %d readouts already measured, %d tasks"
                  % (src, self.n_metrics(), sum(len(v) for v in self.data["results"].values())))
        redo = list(self.data["config"].get("redo") or [])
        if redo:
            self.drop(redo)
        self.save()

    def drop(self, readouts):
        """Forget named readouts so a resumed run measures them again.

        A carried-forward number that turned out to be measured wrongly has to go, or the resume
        guard will protect it forever. Naming it in `CFG["redo"]` removes it from every task and
        every slice, and the removal is recorded in the notes rather than done silently.
        """
        gone = 0
        for suite in self.data["results"].values():
            for rs in suite.values():
                for name in list(rs):
                    if name in readouts:
                        del rs[name]
                        gone += 1
        self.note("redo: dropped %d carried-forward readouts %s" % (gone, sorted(set(readouts))))

    def n_metrics(self):
        return sum(len([k for k in rs if k != "_meta"])
                   for suite in self.data["results"].values() for rs in suite.values())

    def has(self, suite, key, *readouts):
        """True when every named readout for this task is already in the file.

        The resume guards ask this before spending GPU time, so they name the readout a phase
        writes last: if that one is present the whole phase finished.
        """
        rs = (self.data["results"].get(suite) or {}).get(key) or {}
        return bool(readouts) and all(r in rs for r in readouts)

    def note(self, text):
        self.data["notes"].append(text)
        print("note:", text, flush=True)

    def put(self, suite, key, readout, metrics):
        self.data["results"].setdefault(suite, {}).setdefault(key, {})[readout] = metrics
        self.save()

    def meta(self, suite, key, meta):
        self.data["results"].setdefault(suite, {}).setdefault(key, {})["_meta"] = meta
        self.save()

    def phase(self, name, **kw):
        self.data["phases"][name] = dict(kw, at=time.strftime("%H:%M:%S", time.gmtime()),
                                         memory=bc.memory_gb())
        self.save()
        print("phase %-24s %s" % (name, bc.memory_line()), flush=True)

    def save(self):
        tmp = self.path + ".tmp"
        with open(tmp, "w") as fh:
            json.dump(bc.jsonable(self.data), fh, indent=1)
        os.replace(tmp, self.path)


def _progress(label, i, n, t0):
    """One line per block of rows, carrying the host memory left.

    The Kaggle log is the only thing left after a kernel is killed, so the headroom goes on the
    progress line: a run that creeps towards zero over an hour and then dies says what happened,
    and a run that dies with room to spare says it was something else.
    """
    rate = (time.time() - t0) / max(i, 1)
    head = bc.headroom_gb()
    print("    %s %d/%d  %.0f ms/row  eta %.0f s%s"
          % (label, i, n, rate * 1000, rate * (n - i),
             "" if head is None else "  free %.1f GB" % head), flush=True)


def run_records(label, items, fn, budget, every=25):
    """Apply one readout to every item, printing progress and stopping if the clock runs out."""
    out, t0 = [], time.time()
    for i, it in enumerate(items, 1):
        if not budget.ok(10):
            print("    %s stopped early at %d/%d: out of time" % (label, i, len(items)), flush=True)
            break
        try:
            out.append(fn(it))
        except Exception as exc:
            out.append({"id": it["id"], "gold": it["gold"], "pred": None, "probs": None,
                        "error": "%s: %s" % (type(exc).__name__, str(exc)[:200])})
        if i % every == 0 or i == len(items):
            _progress(label, i, len(items), t0)
    return out


def question_of(item, shared):
    return shared if shared is not None else item["question"]


def labels_of(item, shared):
    return shared if shared is not None else item["labels"]


# ----------------------------------------------------------------------------- Vega phases
def vega_task(R, suite, key, items, fit_items, shared_q, shared_labels, qtype, vr, cfg, budget,
              do_heads=True, do_flip=False, views=()):
    """Every Vega readout for one task, under the model's own name."""
    tag = "vega-%s" % vr.size
    chance = None
    if shared_labels:
        chance = 1.0 / max(len(shared_labels), 2)

    def add(name, records, extra=None):
        m = bc.score_metrics(records, shared_labels, qtype, chance)
        if extra:
            m.update(extra)
        R.put(suite, key, "%s/%s" % (tag, name), m)
        return m

    eng = run_records("%s engine" % tag, items,
                      lambda it: vr.engine_record(it, question_of(it, shared_q),
                                                  labels_of(it, shared_labels)), budget)
    for rec, it in zip(eng, items):
        if not shared_labels:
            rec["labels"] = it["labels"]
    add("engine", eng)

    if cfg.get("run_no_adapter", True) and vr.adapter_names and budget.ok(60):
        vr.set_adapters("off")
        try:
            recs = run_records("%s engine_no_adapter" % tag, items,
                               lambda it: vr.engine_record(it, question_of(it, shared_q),
                                                           labels_of(it, shared_labels)), budget)
        finally:
            vr.set_adapters("auto")
        for rec, it in zip(recs, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        add("engine_no_adapter", recs)

    for v in views:
        if not budget.ok(120):
            break
        recs = run_records("%s engine_views%d" % (tag, v), items,
                           lambda it: vr.engine_record(it, question_of(it, shared_q),
                                                       labels_of(it, shared_labels), views=v),
                           budget)
        for rec, it in zip(recs, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        add("engine_views%d" % v, recs)

    if do_flip and budget.ok(60):
        def flipped(it):
            q = dict(question_of(it, shared_q))
            if q["type"] == "choice":
                q["criteria"] = {k: q["criteria"][k] for k in reversed(list(q["criteria"]))}
            elif q["type"] == "score":
                return vr.engine_record(it, q, labels_of(it, shared_labels))
            return vr.engine_record(it, q, labels_of(it, shared_labels))
        recs = run_records("%s engine_reversed_options" % tag, items, flipped, budget)
        for rec, it in zip(recs, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        by = {r["id"]: r for r in recs}
        flips = [1.0 for r in eng if r["id"] in by and by[r["id"]]["pred"] != r["pred"]]
        n_cmp = len([r for r in eng if r["id"] in by])
        add("engine_reversed_options", recs,
            {"option_order_flip_rate": round(len(flips) / n_cmp, 4) if n_cmp else None,
             "option_order_compared": n_cmp})

    if not (do_heads and fit_items and cfg.get("run_ttt", True) and budget.ok(120)):
        return
    # One read per fitting row, one per evaluation row, then every head budget scored off the same
    # features. The learning curve is therefore nearly free once the features exist.
    t0 = time.time()
    Xf = vr.features_for(fit_items, shared_q)
    yf = [it["gold"] for it in fit_items]
    fit_seconds = time.time() - t0
    labels_for_head = shared_labels or fit_items[0]["labels"]
    ladder = RO.fit_head_ladder(Xf, yf, labels_for_head, cfg["shots"], folds=cfg.get("folds", 5))
    R.put(suite, key, "%s/ttt_fit_report" % tag,
          {"fit_rows": len(fit_items), "fit_seconds": round(fit_seconds, 1),
           "shots": {str(k): {kk: vv for kk, vv in v.items() if kk != "head"}
                     for k, v in ladder.items()}})
    Xe = vr.features_for(items, shared_q)
    best_k, best_info = None, None
    for k in sorted(ladder):
        info = ladder[k]
        if not info.get("fitted"):
            continue
        head = info["head"]
        recs = [RO.head_record(head, x, it, labels_for_head,
                               allowed=labels_of(it, shared_labels)) for x, it in zip(Xe, items)]
        for rec, it in zip(recs, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        add("ttt_%d" % k, recs, {"cv_accuracy": info.get("cv_accuracy"),
                                 "classes_in_fit": info.get("classes_seen"),
                                 "beats_chance": info.get("beats_chance")})
        best_k, best_info = k, info
    if best_info is not None:
        head = best_info["head"]
        auto = []
        for x, it, e in zip(Xe, items, eng):
            hr = RO.head_record(head, x, it, labels_for_head,
                                allowed=labels_of(it, shared_labels))
            auto.append(RO.auto_record(e, hr, best_info))
        for rec, it in zip(auto, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        add("auto", auto, {"head_used": bool(best_info.get("beats_chance")), "head_shots": best_k})


# ----------------------------------------------------------------------------- base phases
def base_task(R, suite, key, items, shared_q, shared_labels, qtype, base, cfg, budget):
    tag = "base-%s" % cfg["base_tag"]
    chance = 1.0 / max(len(shared_labels), 2) if shared_labels else None

    def add(name, records, extra=None):
        m = bc.score_metrics(records, shared_labels, qtype, chance)
        if extra:
            m.update(extra)
        R.put(suite, key, "%s/%s" % (tag, name), m)

    if cfg.get("run_base_logprob", True) and budget.ok(60) \
            and not R.has(suite, key, "%s/logprob" % tag):
        pairs = run_records("%s logprob" % tag, items,
                            lambda it: base.logprob_record(it, question_of(it, shared_q),
                                                            labels_of(it, shared_labels)), budget)
        mean_recs = [p[0] if isinstance(p, tuple) else p for p in pairs]
        sum_recs = [p[1] if isinstance(p, tuple) else p for p in pairs]
        for recs in (mean_recs, sum_recs):
            for rec, it in zip(recs, items):
                if not shared_labels:
                    rec["labels"] = it["labels"]
        add("logprob", mean_recs)
        add("logprob_unnormalised", sum_recs)

    if cfg.get("run_base_generate", True) and budget.ok(60) \
            and not R.has(suite, key, "%s/generate" % tag):
        recs = run_records("%s generate" % tag, items,
                           lambda it: base.generate_record(it, question_of(it, shared_q),
                                                            labels_of(it, shared_labels),
                                                            max_new_tokens=cfg.get("gen_tokens", 12)),
                           budget)
        for rec, it in zip(recs, items):
            if not shared_labels:
                rec["labels"] = it["labels"]
        parsed = [r for r in recs if r.get("parsed")]
        # parse_rate alone cannot tell "the weights answered something unusable" from "the prompt
        # left a reasoning block open and the budget went on thinking", and the first full run was
        # the second of those. These three say which.
        lens = [r.get("raw_len") for r in recs if r.get("raw_len") is not None]
        add("generate", recs,
            {"parse_rate": round(len(parsed) / len(recs), 4) if recs else None,
             "raw_len_mean": round(float(sum(lens)) / len(lens), 1) if lens else None,
             "thinking_open_frac": round(sum(1 for r in recs if r.get("thinking_open")) / len(recs),
                                         4) if recs else None})


# ----------------------------------------------------------------------------- Jev phase
def jev_task(R, suite, key, items, shared_q, shared_labels, qtype, jev, cfg, budget):
    recs = jev.run(items, lambda it: question_of(it, shared_q),
                   lambda it: labels_of(it, shared_labels),
                   workers=cfg.get("jev_workers", 6), budget=budget)
    for rec, it in zip(recs, items):
        if not shared_labels:
            rec["labels"] = it["labels"]
    chance = 1.0 / max(len(shared_labels), 2) if shared_labels else None
    m = bc.score_metrics(recs, shared_labels, qtype, chance)
    R.put(suite, key, "jev/%s" % jev.model, m)
    return recs


def free_cuda():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def free_vega(VEGA, R, cache_paths=()):
    """Free the loaded Vega engines, for when nothing after this point reads them.

    The base phase runs last and the summary tables work off the saved metrics, so the two
    engines can go before a second copy of a backbone is loaded. That hands back a whole card and
    takes their weights out of the host accounting, which is the thing the first full run ran out
    of.
    """
    for size, vr in list(VEGA.items()):
        try:
            vr.free()
        except Exception as exc:
            R.note("could not free vega-%s: %s: %s" % (size, type(exc).__name__, str(exc)[:120]))
    free_cuda()
    hinted = bc.drop_file_cache(*[p for p in cache_paths if p])
    R.data["environment"]["memory_after_freeing_vega"] = bc.memory_gb()
    R.save()
    print("freed the Vega engines%s   %s"
          % ("" if not hinted else ", hinted %.1f GB of file cache" % hinted,
             bc.memory_line()), flush=True)


def load_base(repo, device, cfg, R, tag):
    """A second copy of the backbone, loaded only if the host can still hold it.

    Returns None rather than risking the kernel. Host memory, not GPU memory, is what ends a run
    here: a CUDA allocation that does not fit raises an exception this harness catches and
    records, while a host allocation that does not fit has the process killed, and a killed
    kernel takes the remaining phases with it.
    """
    need = float(cfg.get("base_min_free_gb", 6.0))
    head = bc.headroom_gb()
    if head is not None and head < need:
        R.note("did not load the base backbone for %s: %.1f GB of host memory free, %.1f needed"
               % (tag, head, need))
        return None
    print("loading base backbone %s on %s   %s" % (repo, device, bc.memory_line()), flush=True)
    t0 = time.time()
    try:
        base = RO.BaseLM(repo, device, max_len=cfg["base_max_len"], batch=cfg["base_batch"],
                         token=os.environ.get("HF_TOKEN"),
                         thinking=cfg.get("base_thinking", False))
    except Exception as exc:
        R.note("base backbone %s failed to load: %s: %s" % (tag, type(exc).__name__,
                                                            str(exc)[:200]))
        free_cuda()
        return None
    dt = time.time() - t0
    R.data["environment"].setdefault("base_models", {})[tag] = {
        "repo": repo, "device": str(device), "max_len": cfg["base_max_len"],
        "load_seconds": round(dt, 1), "load_mode": base.load_mode,
        "load_failures": getattr(base, "load_failures", []),
        "chat_template": bool(getattr(base.tok, "chat_template", None)),
        "thinking": base.thinking,
        "template_takes_thinking": base.template_takes_thinking,
        "memory_before": base.mem_before, "memory_after": base.mem_after}
    R.save()
    print("loaded in %.1f s via %s   %s" % (dt, base.load_mode, bc.memory_line()), flush=True)
    return base


def free_base(base, cache_paths=()):
    if base is not None:
        base.free()
    free_cuda()
    bc.drop_file_cache(*[p for p in cache_paths if p])
    print("freed the base backbone   %s" % bc.memory_line(), flush=True)


def sliced(items, records, keyfn, qtype="mixed", min_rows=12):
    """Metrics for the whole set and for each slice of it (by primitive, by domain, by tier).

    Slices below `min_rows` are reported with their row count and no metrics, because an accuracy
    over eight rows has a confidence interval wider than any difference it could show.
    """
    by_id = {r["id"]: r for r in records}
    groups = {"all": []}
    for it in items:
        rec = by_id.get(it["id"])
        if rec is None:
            continue
        groups["all"].append(rec)
        name = str(keyfn(it))
        if name != "all":
            groups.setdefault(name, []).append(rec)
    out = {}
    for name, recs in groups.items():
        if len(recs) < min_rows and name != "all":
            out[name] = {"n": len(recs), "too_few_rows": True}
            continue
        out[name] = bc.score_metrics(recs, None, qtype)
    return out


def summarise(data, suite="public", skip=("_meta",)):
    """Macro averages per readout across a suite's tasks: the table to read first.

    Macro, not micro: a 151-class task and a 2-class task count the same, which is the only way a
    mean over tasks of different widths says anything. `skill` is the chance-corrected accuracy, so
    it is the column to compare across tasks rather than raw accuracy.
    """
    import numpy as np

    rows = {}
    for key, readouts in (data.get("results", {}).get(suite) or {}).items():
        for name, m in readouts.items():
            if name in skip or not isinstance(m, dict) or "accuracy" not in m:
                continue
            rows.setdefault(name, []).append((key, m))
    out = {}
    for name, pairs in sorted(rows.items()):
        pick = lambda f: [m[f] for _, m in pairs if m.get(f) is not None]
        mean = lambda f: round(float(np.mean(pick(f))), 4) if pick(f) else None
        out[name] = {"tasks": len(pairs), "accuracy_macro": mean("accuracy"),
                     "skill_macro": mean("skill"), "coverage_macro": mean("coverage"),
                     "ece_macro": mean("ece"), "nll_macro": mean("nll"),
                     "brier_macro": mean("brier"), "aurc_macro": mean("aurc"),
                     "latency_ms_p50_median": round(float(np.median(pick("latency_ms_p50"))), 1)
                     if pick("latency_ms_p50") else None,
                     "macro_f1": mean("macro_f1")}
    return out


def head_to_head(data, suite, a, b):
    """Task by task, which of two readouts won. A mean of means can hide a readout that wins big on
    two tasks and loses everywhere else, and a win count cannot."""
    wins = {"a": 0, "b": 0, "tie": 0, "tasks": [], "a_name": a, "b_name": b}
    for key, readouts in (data.get("results", {}).get(suite) or {}).items():
        ma, mb = readouts.get(a), readouts.get(b)
        if not (isinstance(ma, dict) and isinstance(mb, dict)):
            continue
        if ma.get("accuracy") is None or mb.get("accuracy") is None:
            continue
        d = round(ma["accuracy"] - mb["accuracy"], 4)
        wins["tasks"].append({"task": key, "a": ma["accuracy"], "b": mb["accuracy"], "delta": d})
        wins["a" if d > 0 else ("b" if d < 0 else "tie")] += 1
    wins["tasks"].sort(key=lambda r: r["delta"])
    return wins


## 4. Load the models

0.8B on the first T4, 4B on the second, so both stay resident.

In [ ]:
import json
import platform
import sys
import time

import numpy as np
import torch
import transformers

sys.path.insert(0, os.getcwd())
import bench_common as bc
import bench_jev
import bench_readouts as RO
import bench_run as BR
import bench_suites as BS
import bench_tasks as BT
import vegaml
from bench_load import load_task
from bench_tasks import BY_KEY

R = BR.Results(CFG, CFG["out_path"], resume_from=CFG["resume_from"] or None)
BUDGET = bc.Budget(CFG["budget_hours"] * 3600)
print(bc.memory_line("at the start"))

gpus = []
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    gpus.append({"index": i, "name": p.name, "total_gb": round(p.total_memory / 1024 ** 3, 2),
                 "capability": "%d.%d" % (p.major, p.minor)})
R.data["environment"] = {
    "python": platform.python_version(), "platform": platform.platform(),
    "torch": torch.__version__, "transformers": transformers.__version__,
    "vegaml": getattr(vegaml, "__version__", "unknown"), "numpy": np.__version__,
    "cuda": torch.version.cuda, "gpus": gpus, "n_tasks_registered": len(BT.TASKS)}
print(json.dumps(R.data["environment"], indent=1))

# T4 is capability 7.5, so there is no bf16 here: the backbone runs in fp16 and the engine in fp32,
# which is the same arrangement the published figures were measured under.
#
# One card per size when there are two, both on the first card when there is one, and CPU when
# there is none. CPU works and is unusably slow for the full menu, so it says so rather than
# spending an hour finding out: a notebook started without the accelerator set is the likeliest
# way for someone to meet this.
n_gpu = torch.cuda.device_count()
DEVS = {size: ("cuda:%d" % min(i, n_gpu - 1) if n_gpu else "cpu")
        for i, size in enumerate(CFG["models"])}
print("placement:", DEVS)
if not n_gpu:
    print("no CUDA device: set Accelerator to GPU T4 x2 in the notebook settings. On CPU the full "
          "menu is far beyond the session limit, so run CFG['quick'] = True if you continue.")
elif n_gpu < len(CFG["models"]):
    print("%d card for %d model sizes, so they share it. The base phase frees the engines first, "
          "so it still fits." % (n_gpu, len(CFG["models"])))

VEGA = {}
for size in CFG["models"]:
    t0 = time.time()
    VEGA[size] = RO.VegaRunner(size, DEVS[size], token=os.environ.get("HF_TOKEN"))
    dt = time.time() - t0
    vr = VEGA[size]
    print("vega %-5s on %-7s in %5.1f s  backbone=%s  adapters=%s  max_len=%d"
          % (size, DEVS[size], dt, vr.cfg.get("backbone_name"), vr.adapter_names,
             vr.cfg.get("max_len", 0)), flush=True)
    R.data["environment"].setdefault("models", {})[size] = {
        "device": DEVS[size], "load_seconds": round(dt, 1),
        "backbone": vr.cfg.get("backbone_name"), "adapters": vr.adapter_names,
        "max_len": vr.cfg.get("max_len"), "opt_max_tokens": vr.cfg.get("opt_max_tokens")}
R.data["environment"]["memory_after_vega"] = bc.memory_gb()
R.save()
for i in range(torch.cuda.device_count()):
    print("cuda:%d allocated %.2f GB" % (i, torch.cuda.memory_allocated(i) / 1024 ** 3))
# Host memory, not GPU memory, is what ends a run on Kaggle: a CUDA allocation that does not fit
# raises, and this harness records that, while a host allocation that does not fit has the kernel
# killed with no traceback at all. Both Vega copies are resident from here on, so this is the
# baseline every later load has to fit on top of.
print(bc.memory_line("after loading both Vega instances"))

## 5. Probe the Jev endpoint

One request, printed raw. The response shape is what the parser reads, so seeing it here means a
shape change shows up as one odd cell rather than as a silently wrong column.

In [ ]:
JEV = None
JEV_CACHE = "/kaggle/working/jev_cache.jsonl"
if CFG["jev_cache_from"] and os.path.exists(CFG["jev_cache_from"]):
    import shutil
    if os.path.abspath(CFG["jev_cache_from"]) != os.path.abspath(JEV_CACHE):
        shutil.copyfile(CFG["jev_cache_from"], JEV_CACHE)
    with open(JEV_CACHE) as fh:
        print("seeded the Jev cache with %d responses from %s"
              % (sum(1 for _ in fh), CFG["jev_cache_from"]))
elif CFG["jev_cache_from"]:
    print("jev_cache_from %s does not exist; the hosted column will be paid for again"
          % CFG["jev_cache_from"])

if JEV_KEY:
    JEV = bench_jev.JevClient(JEV_KEY, model=CFG["jev_model"],
                              cache_path=JEV_CACHE, max_rps=CFG["jev_max_rps"])
    probe_q = {"q": {"type": "choice",
                     "instructions": "Which team should own this ticket?",
                     "criteria": {"billing": "payments, invoices and refunds",
                                  "technical": "the product is broken",
                                  "sales": "pricing and new business"}}}
    payload, dt, err = JEV.raw({"message": "We were charged twice this month, please refund it."},
                               probe_q)
    print("latency %.0f ms  error=%s" % (dt or 0, err))
    print(json.dumps(payload, indent=1)[:2000] if payload else "no payload")
    if payload:
        probs, pred = bench_jev.parse_answer((payload.get("answers") or {}).get("q"), "choice",
                                             ["billing", "technical", "sales"])
        print("parsed ->", pred, probs)
        bool_q = {"q": {"type": "boolean", "instructions": "Is this urgent?",
                        "criteria": {"true": "needs attention within the hour",
                                     "false": "it can wait"}}}
        p2, _, e2 = JEV.raw({"message": "Checkout has been down for 40 minutes."}, bool_q)
        print("boolean raw:", json.dumps(p2)[:600], "err", e2)
        R.data["suites"]["jev_probe"] = {"model_reported": payload.get("model"),
                                         "keys": sorted(payload.keys()),
                                         "answer_keys": sorted(((payload.get("answers") or {})
                                                                .get("q") or {}).keys()),
                                         "latency_ms": round(dt or 0, 1)}
    if err:
        print("Jev is unreachable, so the Jev column will be skipped. The local readouts are "
              "unaffected.")
        JEV = None
R.save()

## 6. Smoke test

Four rows through every readout on the 0.8B. If anything in the chain is wrong this fails in a
minute rather than in hour six.

In [ ]:
smoke_key = "boolq"
smoke = load_task(BY_KEY[smoke_key], n_eval=4, n_fit=12, seed=CFG["seed"],
                  cache_dir=os.environ["HF_DATASETS_CACHE"])
assert not smoke.get("skipped"), smoke
vr = VEGA[CFG["models"][0]]
q, labels = smoke["question"], smoke["labels"]
it = smoke["items"][0]
print("state:", json.dumps(it["state"])[:200], "\ngold:", it["gold"])
rec = vr.engine_record(it, q, labels)
print("engine:", {k: rec[k] for k in ("pred", "probs", "abstain", "latency_ms", "set_size")})
X = vr.features_for(smoke["fit"], q)
print("feature vector:", np.asarray(X[0]).shape)
ladder = RO.fit_head_ladder(X, [f["gold"] for f in smoke["fit"]], labels, [6, 12])
print("heads:", {k: {kk: vv for kk, vv in v.items() if kk != "head"} for k, v in ladder.items()})
if ladder.get(12, {}).get("fitted"):
    print("ttt:", RO.head_record(ladder[12]["head"], vr.feature(it, q), it, labels))
print("budget audit:", bc.budget_audit(vr.tok, [q]))
if JEV:
    print("jev:", JEV.record(it, q, labels))

## 7. JevBench public tiers

Fetched from the repository at run time. Engine readouts and the hosted reference only: no head is
fitted on these rows. Sliced by tier, family and answer type, plus paraphrase consistency, which
asks whether the same rule in a different wording gets the same answer.

In [ ]:
JB_RECORDS = {}
jb_items, jb_meta = [], {}


def jb_slices(name, recs):
    """Metrics for this readout over the whole set and over each tier, family and answer type."""
    for keyname, fn in (("all", lambda it: "all"),
                        ("tier", lambda it: "tier:" + str(it["tier"])),
                        ("family", lambda it: "family:" + str(it["family"])),
                        ("qtype", lambda it: "qtype:" + str(it["qtype"]))):
        for slice_name, m in BR.sliced(jb_items, recs, fn).items():
            if keyname != "all" and slice_name == "all":
                continue
            R.put("jevbench", slice_name, name, m)
    cons = BS.paraphrase_consistency(jb_items, recs)
    if cons:
        R.put("jevbench", "paraphrase", name, cons)


if CFG["jevbench"] and BUDGET.ok(300):
    jb_items, jb_meta = BS.fetch_jevbench()
    R.data["suites"]["jevbench"] = jb_meta
    R.save()
    print(json.dumps(jb_meta, indent=1))
    if CFG["quick"]:
        jb_items = jb_items[:12]

    for size, vr in VEGA.items():
        if not BUDGET.ok(120):
            break
        if R.has("jevbench", "all", "vega-%s/engine" % size):
            print("jevbench vega-%s: already in the resumed file" % size, flush=True)
            continue
        recs = BR.run_records("jevbench vega-%s engine" % size, jb_items,
                              lambda it: vr.engine_record(it, it["question"], it["labels"]), BUDGET)
        for r, it in zip(recs, jb_items):
            r["labels"] = it["labels"]
        JB_RECORDS["vega-%s/engine" % size] = recs
        jb_slices("vega-%s/engine" % size, recs)
        if CFG["run_no_adapter"] and vr.adapter_names and BUDGET.ok(120):
            vr.set_adapters("off")
            try:
                recs = BR.run_records("jevbench vega-%s no_adapter" % size, jb_items,
                                      lambda it: vr.engine_record(it, it["question"],
                                                                  it["labels"]), BUDGET)
            finally:
                vr.set_adapters("auto")
            for r, it in zip(recs, jb_items):
                r["labels"] = it["labels"]
            jb_slices("vega-%s/engine_no_adapter" % size, recs)

    if JEV and BUDGET.ok(120) and not R.has("jevbench", "all", "jev/%s" % JEV.model):
        recs = JEV.run(jb_items, lambda it: it["question"], lambda it: it["labels"],
                       workers=CFG["jev_workers"], budget=BUDGET)
        for r, it in zip(recs, jb_items):
            r["labels"] = it["labels"]
        JB_RECORDS["jev"] = recs
        jb_slices("jev/%s" % JEV.model, recs)
    R.phase("jevbench", items=len(jb_items), readouts=sorted(JB_RECORDS))
    print(json.dumps(R.data["results"].get("jevbench", {}).get("all", {}), indent=1))

## 8. The 43 public tasks

Loaded first, so the row counts, licences and the option budget audit are recorded before any GPU
time is spent. A task that cannot be loaded (a dataset moved, gated, or still script-only) is
recorded with the reason and skipped rather than silently dropped.

In [ ]:
TASK_KEYS = CFG["task_keys"] or [t["key"] for t in BT.TASKS]
LOADED = {}
tok0 = VEGA[CFG["models"][0]].tok
cap = VEGA[CFG["models"][0]].cfg.get("opt_max_tokens", 32)

for key in TASK_KEYS:
    t = BY_KEY[key]
    t0 = time.time()
    try:
        out = load_task(t, n_eval=CFG["n_eval"], n_fit=CFG["n_fit"], seed=CFG["seed"],
                        cache_dir=os.environ["HF_DATASETS_CACHE"])
    except Exception as exc:
        out = {"skipped": "%s: %s" % (type(exc).__name__, str(exc)[:200])}
    if out.get("skipped"):
        R.meta("public", key, {"skipped": out["skipped"], "hf": t["hf"], "config": t["config"],
                               "group": t["group"], "licence": t["licence"]})
        print("%-24s SKIPPED  %s" % (key, out["skipped"]), flush=True)
        continue
    LOADED[key] = out
    qs = [out["question"]] if out["kind"] == "fixed" else [i["question"] for i in out["items"][:40]]
    audit = bc.budget_audit(tok0, qs, opt_max_tokens=cap)
    meta = dict(out["meta"], n_eval=len(out["items"]), n_fit=len(out["fit"]),
                kind=out["kind"], labels=len(out["labels"]),
                chance=round(1.0 / max(len(out["labels"]), 2), 4),
                budget_audit=audit,
                trivial=bc.trivial_baselines([i["gold"] for i in out["items"]], out["labels"]),
                load_seconds=round(time.time() - t0, 1))
    R.meta("public", key, meta)
    print("%-24s %-7s labels=%-4d eval=%-4d fit=%-4d opt_p95=%-4d over_cap=%4.0f%% collide=%d"
          % (key, out["kind"], len(out["labels"]), len(out["items"]), len(out["fit"]),
             audit["option_tokens_p95"], 100 * audit["options_over_cap_frac"],
             audit["truncation_collisions"]), flush=True)

print("\nloaded %d of %d tasks" % (len(LOADED), len(TASK_KEYS)))
R.phase("load_public", loaded=len(LOADED), requested=len(TASK_KEYS))

### 8a. Vega readouts, both sizes

Per task and per size: the engine, the engine with adapters off, surface-form averaging on a
subset, a reversed-option run on a subset to measure order sensitivity, then the head ladder and
`auto`. Fitting rows come from the task's own training split and are never scored.

In [ ]:
for size in CFG["models"]:
    vr = VEGA[size]
    # The readouts a finished task carries, so a resumed run can tell done from half done. The
    # fit report is written after the whole head ladder, the engine read before it.
    done = ["vega-%s/engine" % size] + (["vega-%s/ttt_fit_report" % size]
                                        if CFG.get("run_ttt", True) else [])
    todo = [k for k in LOADED if not R.has("public", k, *done)]
    if len(todo) < len(LOADED):
        print("vega-%s: %d of %d tasks already in the resumed file"
              % (size, len(LOADED) - len(todo), len(LOADED)), flush=True)
    for i, key in enumerate(todo, 1):
        out = LOADED[key]
        if not BUDGET.ok(120):
            R.note("stopped the vega-%s public phase at task %d of %d: out of budget"
                   % (size, i, len(todo)))
            break
        print("\n[%d/%d] %s  vega-%s  (%.0f min used)"
              % (i, len(todo), key, size, BUDGET.spent() / 60), flush=True)
        BR.vega_task(R, "public", key, out["items"], out["fit"],
                     out.get("question"), out["labels"], out["meta"]["qtype"], vr, CFG, BUDGET,
                     do_heads=True, do_flip=key in CFG["flip_tasks"],
                     views=[v for v in CFG["views"] if key in CFG["views_tasks"]])
    R.phase("public_vega_%s" % size, tasks=len(todo), minutes=round(BUDGET.spent() / 60, 1))
BR.free_cuda()

### 8b. The hosted reference on the same tasks

Same questions, same rows, over the network. Responses are cached on disk, so a re-run costs
nothing and returns the same answers.

In [ ]:
if JEV and CFG["jev_on_public"]:
    todo = [k for k in LOADED if not R.has("public", k, "jev/%s" % JEV.model)]
    for i, key in enumerate(todo, 1):
        out = LOADED[key]
        if not BUDGET.ok(60):
            R.note("stopped the jev public phase at task %d of %d: out of budget" % (i, len(todo)))
            break
        t0 = time.time()
        recs = BR.jev_task(R, "public", key, out["items"], out.get("question"), out["labels"],
                           out["meta"]["qtype"], JEV, CFG, BUDGET)
        m = R.data["results"]["public"][key]["jev/%s" % JEV.model]
        print("[%d/%d] %-24s jev acc=%.3f cov=%.3f  %.0f s"
              % (i, len(todo), key, m.get("accuracy", 0), m.get("coverage", 0),
                 time.time() - t0), flush=True)
    R.data["suites"]["jev_client"] = JEV.summary()
    R.phase("public_jev", minutes=round(BUDGET.spent() / 60, 1))
    print(json.dumps(JEV.summary(), indent=1))

## 9. DecisionBench

43 tasks, three primitives, gold distributions as well as gold labels. Engine readouts, the base
model and the hosted reference: nothing is fitted here either.

Its ordinal rows are asked as a **choice**, not as a score. Their candidate lists are shuffled per
row and the stored `ordinal_value` follows that shuffled position rather than the severity of the
label, so one row runs Negligible to Critical and the next runs the other way. A score-typed
question asserts that level i sits below level i+1, which these rows do not carry.

In [ ]:
DB_RECORDS = {}
db_items, db_meta = [], {}


def db_slices(name, recs):
    """Metrics for this readout over the whole set and over each primitive, domain and task."""
    for fn, prefix in ((lambda it: "all", None),
                       (lambda it: "primitive:" + str(it["primitive"]), "primitive"),
                       (lambda it: "qtype:" + str(it["qtype"]), "qtype"),
                       (lambda it: "reasoning:%s" % it["reasoning_required"], "reasoning"),
                       (lambda it: "domain:" + str(it["domain"]), "domain"),
                       (lambda it: "task:" + str(it["task"]), "task")):
        for slice_name, m in BR.sliced(db_items, recs, fn).items():
            if prefix and slice_name == "all":
                continue
            R.put("decisionbench", slice_name, name, m)
    kl = BS.soft_gold_kl(db_items, recs)
    if kl:
        R.put("decisionbench", "soft_gold", name, kl)


if BUDGET.ok(300):
    try:
        db_items, db_meta = BS.fetch_decisionbench(
            per_task=CFG["decisionbench_per_task"], seed=CFG["seed"],
            cache_dir=os.environ["HF_DATASETS_CACHE"])
    except Exception as exc:
        db_meta = {"error": "%s: %s" % (type(exc).__name__, str(exc)[:200])}
    R.data["suites"]["decisionbench"] = db_meta
    R.save()
    print(json.dumps({k: v for k, v in db_meta.items() if k != "ordinal_note"}, indent=1))

    for size, vr in VEGA.items():
        if not BUDGET.ok(180) or not db_items:
            break
        if R.has("decisionbench", "all", "vega-%s/engine" % size):
            print("decisionbench vega-%s: already in the resumed file" % size, flush=True)
            continue
        recs = BR.run_records("decisionbench vega-%s engine" % size, db_items,
                              lambda it: vr.engine_record(it, it["question"], it["labels"]), BUDGET)
        for r, it in zip(recs, db_items):
            r["labels"] = it["labels"]
        DB_RECORDS["vega-%s/engine" % size] = recs
        db_slices("vega-%s/engine" % size, recs)

    if JEV and CFG["jev_on_decisionbench"] and db_items and BUDGET.ok(180) \
            and not R.has("decisionbench", "all", "jev/%s" % JEV.model):
        recs = JEV.run(db_items, lambda it: it["question"], lambda it: it["labels"],
                       workers=CFG["jev_workers"], budget=BUDGET)
        for r, it in zip(recs, db_items):
            r["labels"] = it["labels"]
        DB_RECORDS["jev"] = recs
        db_slices("jev/%s" % JEV.model, recs)
    R.phase("decisionbench", items=len(db_items), readouts=sorted(DB_RECORDS))
    print(json.dumps(R.data["results"].get("decisionbench", {}).get("all", {}), indent=1))

## 10. The base model on both suites

The same frozen weights the engine reads, loaded a second time because Vega's perception registers
forward hooks that stop the forward pass at the deepest layer it needs, which makes that instance
unusable for generation. The backbone is a base checkpoint rather than an instruction tune, so
`generate`'s **parse rate** matters as much as its accuracy: it says whether the weights can be
used this way at all. Reasoning is switched off in the chat template for both sizes, because the
0.8B and 4B templates default the other way round and the open block on the 4B spent the whole
generation budget thinking; an open block that never closes now counts as unanswered rather than
having an option read out of the reasoning.

One load per size, used for the public tasks and for DecisionBench before it is freed. The first
full run loaded the 4B backbone twice, once per suite, and the second load is where the kernel was
killed: 9.3 GB of weights materialised in host memory on top of two resident Vega instances. The
second copy bought nothing, since these are the same weights answering a different suite, so the
two phases now share one load, the load streams its shards straight to the card, and the Vega
engines are freed first because nothing after this cell reads them. If the host is still short of
room the load is skipped with a note rather than risking the kernel.

In [ ]:
BASE_DONE = {}
# what a finished base phase leaves behind for one task, so a resumed run can tell done from
# half done. base_task guards each of these on its own as well, so a redo of one readout does
# not pay for the other.
WANT_BASE = ([] + (["logprob"] if CFG.get("run_base_logprob", True) else [])
             + (["generate"] if CFG.get("run_base_generate", True) else []))
REPOS = {size: VEGA[size].cfg.get("backbone_name") for size in CFG["models"]}
CACHE_DIRS = [os.environ.get("HF_HOME")]


def base_todo(size):
    want = ["base-%s/%s" % (size, w) for w in WANT_BASE]
    return ([k for k in LOADED if not R.has("public", k, *want)],
            bool(db_items) and not R.has("decisionbench", "all", *want))


# Nothing past this cell reads the Vega engines: the summary tables work off the saved metrics.
# Freeing them here gives the base copy a card to itself and takes their weights out of the
# host's accounting, which is where the first full run ran out of room.
if CFG.get("free_vega_before_base", True) \
        and any(pub or db for pub, db in (base_todo(s) for s in CFG["models"])):
    BR.free_vega(VEGA, R, CACHE_DIRS)

for size in CFG["models"]:
    if not BUDGET.ok(300):
        R.note("skipped the base-model baselines for %s: out of budget" % size)
        break
    repo = REPOS[size]
    dev = DEVS[CFG["models"][0]]          # the lighter card, since Vega 4B holds the other one
    pub_todo, db_todo = base_todo(size)
    if not pub_todo and not db_todo:
        print("base-%s: already measured in the resumed file" % size, flush=True)
        BASE_DONE[size] = "resumed"
        continue
    base = BR.load_base(repo, dev, CFG, R, size)
    if base is None:                      # out of host memory, or the weights would not load
        continue
    cfg_base = dict(CFG, base_tag=size)
    try:
        for i, key in enumerate(pub_todo, 1):
            if not BUDGET.ok(90):
                R.note("stopped the base-%s public phase at task %d of %d: out of budget"
                       % (size, i, len(pub_todo)))
                break
            out = LOADED[key]
            print("\n[%d/%d] %s  base-%s  (%.0f min used)"
                  % (i, len(pub_todo), key, size, BUDGET.spent() / 60), flush=True)
            BR.base_task(R, "public", key, out["items"], out.get("question"), out["labels"],
                         out["meta"]["qtype"], base, cfg_base, BUDGET)
        R.phase("public_base_%s" % size, tasks=len(pub_todo),
                minutes=round(BUDGET.spent() / 60, 1))

        if db_todo and BUDGET.ok(240):
            for name in ("logprob", "generate"):
                if not CFG.get("run_base_%s" % name, True) or not BUDGET.ok(120):
                    continue
                if R.has("decisionbench", "all", "base-%s/%s" % (size, name)):
                    continue
                if name == "logprob":
                    pairs = BR.run_records("decisionbench base-%s logprob" % size, db_items,
                                           lambda it: base.logprob_record(it, it["question"],
                                                                          it["labels"]), BUDGET)
                    recs = [p[0] if isinstance(p, tuple) else p for p in pairs]
                else:
                    recs = BR.run_records("decisionbench base-%s generate" % size, db_items,
                                          lambda it: base.generate_record(it, it["question"],
                                                                          it["labels"],
                                                                          CFG["gen_tokens"]),
                                          BUDGET)
                for r, it in zip(recs, db_items):
                    r["labels"] = it["labels"]
                db_slices("base-%s/%s" % (size, name), recs)
            R.phase("decisionbench_base_%s" % size, minutes=round(BUDGET.spent() / 60, 1))
    finally:
        BR.free_base(base, CACHE_DIRS)    # before the next size is loaded, not after
        base = None
    BASE_DONE[size] = True

## 11. Summary tables

Macro means over tasks, so a 151-class task counts the same as a yes/no one. Read `skill` across
tasks rather than `accuracy`: it is chance-corrected.

In [ ]:
import pandas as pd

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

SUMMARY = {"public_macro": BR.summarise(R.data, "public")}
df = pd.DataFrame(SUMMARY["public_macro"]).T.sort_values("skill_macro", ascending=False)
print("=== public tasks, macro over %d tasks ===" % len(LOADED))
print(df[["tasks", "accuracy_macro", "skill_macro", "coverage_macro", "ece_macro", "nll_macro",
          "latency_ms_p50_median"]])
R.data["summary"] = SUMMARY
R.save()

In [ ]:
# The ablation ladder on one line per readout, and the head learning curve beside the engine.
rows = []
for key, readouts in R.data["results"].get("public", {}).items():
    for name, m in readouts.items():
        if name == "_meta" or not isinstance(m, dict) or "accuracy" not in m:
            continue
        rows.append({"task": key, "readout": name, "group": readouts["_meta"].get("group"),
                     "labels": readouts["_meta"].get("labels"), "accuracy": m["accuracy"],
                     "skill": m.get("skill"), "ece": m.get("ece"), "coverage": m.get("coverage"),
                     "nll": m.get("nll")})
TASKS_DF = pd.DataFrame(rows)
if not TASKS_DF.empty:
    piv = TASKS_DF.pivot_table(index="readout", values=["skill", "accuracy", "ece"],
                               aggfunc="mean").sort_values("skill", ascending=False)
    print("=== every readout, macro ===")
    print(piv)
    print("\n=== head learning curve against the engine, by task width ===")
    curve = TASKS_DF[TASKS_DF.readout.str.contains("ttt_|/engine$", regex=True)]
    if not curve.empty:
        print(curve.pivot_table(index=["labels", "task"], columns="readout",
                                values="accuracy").round(3).head(50))

In [ ]:
# Head to head: where each readout wins, task by task. A mean of means hides this.
for size in CFG["models"]:
    for a, b in ((("vega-%s/engine" % size), ("base-%s/logprob" % size)),
                 (("vega-%s/engine" % size), ("vega-%s/engine_no_adapter" % size)),
                 (("vega-%s/ttt_50" % size), ("vega-%s/engine" % size)),
                 (("vega-%s/engine" % size), ("jev/%s" % (JEV.model if JEV else "none")))):
        h = BR.head_to_head(R.data, "public", a, b)
        if not h["tasks"]:
            continue
        print("%-34s vs %-34s  wins %d / %d, ties %d"
              % (a, b, h["a"], h["a"] + h["b"], h["tie"]))
        worst = h["tasks"][:3]
        best = h["tasks"][-3:]
        print("    biggest losses:", [(t["task"], t["delta"]) for t in worst])
        print("    biggest wins:  ", [(t["task"], t["delta"]) for t in best])
        R.data.setdefault("summary", {}).setdefault("head_to_head", []).append(h)
R.save()

In [ ]:
# JevBench and DecisionBench tables
for suite in ("jevbench", "decisionbench"):
    block = R.data["results"].get(suite, {})
    if not block:
        continue
    print("\n=== %s ===" % suite)
    frames = {}
    for slice_name, readouts in block.items():
        for name, m in readouts.items():
            if not isinstance(m, dict) or "accuracy" not in m:
                continue
            frames.setdefault(name, {})[slice_name] = m["accuracy"]
    dfx = pd.DataFrame(frames)
    cols = [c for c in ["all"] if c in dfx.index] + \
           sorted([c for c in dfx.index if c.startswith("tier:") or c.startswith("primitive:")
                   or c.startswith("qtype:")])
    print(dfx.loc[cols].round(3) if cols else dfx.round(3).head(20))
    detail = {s: {n: {k: v for k, v in m.items() if k in ("accuracy", "skill", "ece", "nll",
                                                          "coverage", "paraphrase_consistency")}
                  for n, m in rs.items() if isinstance(m, dict)}
              for s, rs in block.items() if s in ("all", "paraphrase", "soft_gold")}
    print(json.dumps(detail, indent=1)[:2500])

## 12. Save

One JSON with everything: configuration, environment, per task metadata and licences, the option
budget audit, every readout's metrics, the head learning curves, the robustness runs and the
hosted reference. No dataset text.

In [ ]:
R.data["completed"] = True
R.data["wall_clock_minutes"] = round(BUDGET.spent() / 60, 1)
R.data["budget_exhausted"] = not BUDGET.ok(0)
if JEV:
    R.data["suites"]["jev_client"] = JEV.summary()
R.save()

size_kb = os.path.getsize(CFG["out_path"]) / 1024
n_metrics = sum(len([k for k in rs if k != "_meta"])
                for suite in R.data["results"].values() for rs in suite.values())
print("wrote %s  (%.0f KB, %d metric blocks, %.0f minutes)"
      % (CFG["out_path"], size_kb, n_metrics, R.data["wall_clock_minutes"]))
print("notes:")
for n in R.data["notes"]:
    print(" -", n)
print("\nphases:", json.dumps(R.data["phases"], indent=1))

## 13. How to read the output

* `results.public.<task>._meta` carries the licence, the split, the row counts, the chance level,
  the majority-class baseline and the **option budget audit**. Read the audit before reading a low
  score on CaseHOLD, HellaSwag, MedQA or MMLU-Pro: Vega truncates an option at 32 tokens, and
  `truncation_collisions` counts the cases where two options became the same tokens, which makes
  those rows unanswerable by construction rather than hard.
* `skill` is chance-corrected accuracy. Use it, not `accuracy`, to compare a 2-class task with a
  151-class one.
* `coverage` below 1.0 means a readout could not answer some rows. For `base-*/generate` that is
  the parse rate, and it is the honest headline for prompting a base checkpoint. Read it with
  `raw_len_mean` and `thinking_open_frac` beside it: a parse rate near zero with a long mean
  length and thinking open is a prompt-format result, not a result about the weights.
* `ece`, `nll`, `brier` and `aurc` need probabilities. A readout that only emits a label reports
  them as null rather than as zero.
* `abstain_rate` and `accuracy_when_not_abstaining` are the engine's own refusal to answer.
  `conformal_coverage` is how often the conformal set contained the gold label, which is the number
  the set is calibrated to hold at about 0.9 for alpha 0.1.
* `engine_reversed_options.option_order_flip_rate` is how often reversing the option order changed
  the answer. `engine_views2` is the engine's own remedy for that, averaged over both orders.
* `ttt_fit_report` holds the cross-validated accuracy per head budget, how many classes the fitting
  rows reached, and whether the head beat chance. A head at or below chance is noise; `auto` leaves
  those questions to the engine, and the report says which.
* Jev figures here are measured by this notebook against the live endpoint on these rows. They are
  not the published leaderboard numbers, which use held-out items as well, so compare them in kind
  and not value.
* `phases.*.memory` is the host memory at each phase boundary: this process's resident size, the
  container's own usage against its limit, and the headroom left. `environment.base_models.*`
  records which load path the second copy of the backbone took and what it cost. A run that ends
  with no `completed` flag and no final phase was killed rather than stopped, and these are the
  numbers that say whether memory was why.
* `resumed_from` is present when the run carried an earlier file forward, with that run's config
  and environment under it. Readouts measured in the earlier run are exactly as it measured them,
  which also means they were measured on that run's library versions.
